In [184]:
import pandas as pd
import numpy as np
from pathlib import Path

Data Load

In [229]:
FILE_PATH = Path(
    "../data/wasde/wasde_processed_all_historicals.csv"
)
df = pd.read_csv(FILE_PATH)

df[df["commodity"]=="barley"]["Report Date"].unique()

df[(df["commodity"]=="corn")&(df["table"]=="us")]

,Report Date,Day,Month,Year,Value,commodity,country,snd_item,wheat_class,crop,table,crop year,month offset
144,2024/1/12,12,1,2024,78.700000,corn,united states,harvested area,NaN,oc,us,2022,2_01
145,2024/1/12,12,1,2024,88.200000,corn,united states,planted area,NaN,oc,us,2022,2_01
146,2024/1/12,12,1,2024,1377.000000,corn,united states,beginning stocks,NaN,oc,us,2022,2_01
147,2024/1/12,12,1,2024,12045.000000,corn,united states,domestic total,NaN,oc,us,2022,2_01
148,2024/1/12,12,1,2024,1360.000000,corn,united states,ending stocks,NaN,oc,us,2022,2_01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
200813,2026/9/11,11,9,2026,1567.000000,corn,united states,ending stocks,NaN,nc,us,2026,0_09
200814,2026/9/11,11,9,2026,4.150000,corn,united states,avg. farm price,NaN,oc,us,2025,1_09
200815,2026/9/11,11,9,2026,4.800000,corn,united states,avg. farm price,NaN,nc,us,2026,0_09
202744,2026/9/11,11,9,2026,0.096848,corn,united states,stock to use,NaN,nc,us,2026,0_09


Clean Data

Set Default Filters

In [148]:
commodity = "wheat"
country = "united states"
report = "us"
attribute = "yield"
crop = "nc"

In [149]:
month_names = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}


Filter

In [150]:
filter = df[
    (df["commodity"] == commodity) &
    (df["country"] == country) &
    (df["table"] == report) &
    (df["snd_item"] == attribute) &
    (df["crop"] == crop)
].copy()

Pivot Table

In [151]:
pivot = filter.pivot(
    index = "Year",
    columns = "Month",
    values = "Value"
)

In [152]:
# Rename months
pivot = pivot.rename(
    columns=month_names
)

# Round
pivot = pivot.round(2)

In [153]:
pivot

Month,January,February,March,April,May,June,July,August,September,October,November,December
Year,,,,,,,,,,,,
2010,NaN,NaN,NaN,NaN,43.4,43.9,45.9,46.9,46.9,46.7,46.4,46.4
2011,46.4,46.4,46.4,46.4,42.5,43.1,44.6,45.2,45.2,43.9,43.7,43.7
2012,43.7,43.7,43.7,43.7,45.7,45.4,45.6,46.5,46.5,46.3,46.3,46.3
2013,46.3,46.3,46.3,46.3,44.1,44.6,46.2,46.2,46.2,NaN,47.2,47.2
2014,47.2,47.2,47.2,47.2,42.7,42.3,43.1,43.9,43.9,43.8,43.7,43.7
2015,43.7,43.7,43.7,43.7,43.5,44.2,44.3,44.1,44.1,43.6,43.6,43.6
2016,43.6,43.6,43.6,43.6,46.7,48.6,51.3,52.6,52.6,52.6,52.6,52.6
2017,52.6,52.6,52.6,52.6,47.2,47.3,46.2,45.6,45.6,46.3,46.3,46.3
2018,46.3,46.3,46.3,46.3,46.8,46.9,47.5,47.4,47.4,47.6,47.6,47.6


In [154]:
yoy = filter.copy()

In [155]:
year = filter[["Year", "Month", "Value"]].copy()

previous_year = filter[
    ["Year", "Month", "Value"]
].copy()

previous_year["Year"] = previous_year["Year"] + 1

previous_year = previous_year.rename(
    columns={"Value": "Previous Year Value"}
)

yoy = yoy.merge(
    previous_year,
    on=["Year", "Month"],
    how="left"
)

yoy["YoY Change"] = (
    yoy["Value"] - yoy["Previous Year Value"]
)

In [156]:
df_yoy = yoy.pivot_table(
    index = "Year",
    columns = "Month",
    values = "YoY Change"
)

In [157]:
df_yoy.rename(
    columns = month_names
)


Month,January,February,March,April,May,June,July,August,September,October,November,December
Year,,,,,,,,,,,,
2011,NaN,NaN,NaN,NaN,-0.9,-0.8,-1.3,-1.7,-1.7,-2.8,-2.7,-2.7
2012,-2.7,-2.7,-2.7,-2.7,3.2,2.3,1.0,1.3,1.3,2.4,2.6,2.6
2013,2.6,2.6,2.6,2.6,-1.6,-0.8,0.6,-0.3,-0.3,NaN,0.9,0.9
2014,0.9,0.9,0.9,0.9,-1.4,-2.3,-3.1,-2.3,-2.3,NaN,-3.5,-3.5
2015,-3.5,-3.5,-3.5,-3.5,0.8,1.9,1.2,0.2,0.2,-0.2,-0.1,-0.1
2016,-0.1,-0.1,-0.1,-0.1,3.2,4.4,7.0,8.5,8.5,9.0,9.0,9.0
2017,9.0,9.0,9.0,9.0,0.5,-1.3,-5.1,-7.0,-7.0,-6.3,-6.3,-6.3
2018,-6.3,-6.3,-6.3,-6.3,-0.4,-0.4,1.3,1.8,1.8,1.3,1.3,1.3
2019,NaN,1.3,1.3,1.3,1.8,1.8,2.5,4.2,4.2,4.0,4.1,4.1


In [158]:
month = filter[["Year", "Month", "Value"]].copy()

previous_month = filter[
    ["Year", "Month", "Value"]
].copy()

# Move previous month forward
previous_month["Month"] = previous_month["Month"] + 1

# December -> January of the following year
previous_month.loc[
    previous_month["Month"] == 13, "Month"
] = 1

previous_month.loc[
    previous_month["Month"] == 1,
    "Year"
] = previous_month.loc[
    previous_month["Month"] == 1,
    "Year"
] + 1

previous_month = previous_month.rename(
    columns={"Value": "Previous Month Value"}
)

mom = month.merge(
    previous_month,
    on=["Year", "Month"],
    how="left"
)

mom["MoM Change"] = (
    mom["Value"] - mom["Previous Month Value"]
)

In [159]:
df_mom = mom.pivot_table(
    index="Year",
    columns="Month",
    values="MoM Change"
).rename(
    columns = month_names
)

df_mom

Month,January,February,March,April,May,June,July,August,September,October,November,December
Year,,,,,,,,,,,,
2010,NaN,NaN,NaN,NaN,NaN,0.5,2.0,1.0,0.0,-0.2,-0.3,0.0
2011,0.0,0.0,0.0,0.0,-3.9,0.6,1.5,0.6,0.0,-1.3,-0.2,0.0
2012,0.0,0.0,0.0,0.0,2.0,-0.3,0.2,0.9,0.0,-0.2,0.0,0.0
2013,0.0,0.0,0.0,0.0,-2.2,0.5,1.6,0.0,0.0,NaN,NaN,0.0
2014,0.0,0.0,0.0,0.0,-4.5,-0.4,0.8,0.8,0.0,-0.1,-0.1,0.0
2015,0.0,0.0,0.0,0.0,-0.2,0.7,0.1,-0.2,0.0,-0.5,0.0,0.0
2016,0.0,0.0,0.0,0.0,3.1,1.9,2.7,1.3,0.0,0.0,0.0,0.0
2017,0.0,0.0,0.0,0.0,-5.4,0.1,-1.1,-0.6,0.0,0.7,0.0,0.0
2018,0.0,0.0,0.0,0.0,0.5,0.1,0.6,-0.1,0.0,0.2,0.0,0.0


In [160]:
# Create May value column
may_values = (
    mom[mom["Month"] == 5][["Year", "Value"]]
    .rename(columns={"Value": "May Value"})
)

may_change = mom.merge(
    may_values,
    on="Year",
    how="left"
)

# Calculate change from May
may_change["Change from May"] = (
    may_change["Value"] - may_change["May Value"]
)

# Pivot the changes
df_may_change = may_change.pivot_table(
    index="Year",
    columns="Month",
    values="Change from May"
)

# Put May Value into the table
may_value_series = (
    may_change
    .drop_duplicates("Year")
    .set_index("Year")["May Value"]
)

df_may_change["May Value"] = may_value_series

df_may_change = df_may_change[
    ["May Value"] +
    [col for col in df_may_change.columns if col != "May Value"]
]

df_may_change.rename(
    columns = month_names
)

Month,May Value,January,February,March,April,May,June,July,August,September,October,November,December
Year,,,,,,,,,,,,,
2010,43.4,NaN,NaN,NaN,NaN,0.0,0.5,2.5,3.5,3.5,3.3,3.0,3.0
2011,42.5,3.9,3.9,3.9,3.9,0.0,0.6,2.1,2.7,2.7,1.4,1.2,1.2
2012,45.7,-2.0,-2.0,-2.0,-2.0,0.0,-0.3,-0.1,0.8,0.8,0.6,0.6,0.6
2013,44.1,2.2,2.2,2.2,2.2,0.0,0.5,2.1,2.1,2.1,NaN,3.1,3.1
2014,42.7,4.5,4.5,4.5,4.5,0.0,-0.4,0.4,1.2,1.2,1.1,1.0,1.0
2015,43.5,0.2,0.2,0.2,0.2,0.0,0.7,0.8,0.6,0.6,0.1,0.1,0.1
2016,46.7,-3.1,-3.1,-3.1,-3.1,0.0,1.9,4.6,5.9,5.9,5.9,5.9,5.9
2017,47.2,5.4,5.4,5.4,5.4,0.0,0.1,-1.0,-1.6,-1.6,-0.9,-0.9,-0.9
2018,46.8,-0.5,-0.5,-0.5,-0.5,0.0,0.1,0.7,0.6,0.6,0.8,0.8,0.8


In [173]:
# ============================================================
# MARKETING YEAR
# ============================================================
selected_commodity = "wheat"
marketing_year_start = {
    "corn": 9,
    "soybeans": 9,
    "wheat": 6,
    "cotton": 8,
    "sorghum": 9,
    "barley": 6,
    "oats": 6,
    "rice": 8
}

start_month = marketing_year_start[selected_commodity]


# ============================================================
# PREPARE DATA
# ============================================================

month = filter[
    ["Year", "Month", "Value"]
].copy()


# ============================================================
# CREATE MARKETING YEAR
# ============================================================

month["Marketing Year"] = np.where(
    month["Month"] >= start_month,
    month["Year"],
    month["Year"] - 1
)


# ============================================================
# PREVIOUS MONTH
# ============================================================

previous_month = filter[
    ["Year", "Month", "Value"]
].copy()

previous_month["Month"] = previous_month["Month"] + 1

# December -> January
previous_month.loc[
    previous_month["Month"] == 13,
    "Month"
] = 1

previous_month.loc[
    previous_month["Month"] == 1,
    "Year"
] = previous_month.loc[
    previous_month["Month"] == 1,
    "Year"
] + 1

previous_month = previous_month.rename(
    columns={
        "Value": "Previous Month Value"
    }
)


# ============================================================
# MERGE
# ============================================================

mom = month.merge(
    previous_month,
    on=["Year", "Month"],
    how="left"
)


# ============================================================
# MOM CHANGE
# ============================================================

mom["MoM Change"] = (
    mom["Value"]
    - mom["Previous Month Value"]
)



In [174]:
month_order = (
    list(range(start_month, 13))
    + list(range(1, start_month))
)

In [175]:
df_mom = mom.pivot_table(
    index="Marketing Year",
    columns="Month",
    values="MoM Change"
)

df_mom = df_mom.reindex(
    columns=month_order
)

In [176]:
month_names = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun",
    7: "Jul",
    8: "Aug",
    9: "Sep",
    10: "Oct",
    11: "Nov",
    12: "Dec"
}

df_mom = df_mom.rename(
    columns=month_names
)

In [177]:
df_mom.index = (
    df_mom.index.astype(int).astype(str)
    + "/"
    + (df_mom.index.astype(int) + 1).astype(str).str[-2:]
)

df_mom.index.name = "Marketing Year"

display(df_mom)

Month,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Jan,Feb,Mar,Apr,May
Marketing Year,,,,,,,,,,,,
2010/11,0.5,2.0,1.0,0.0,-0.2,-0.3,0.0,0.0,0.0,0.0,0.0,-3.9
2011/12,0.6,1.5,0.6,0.0,-1.3,-0.2,0.0,0.0,0.0,0.0,0.0,2.0
2012/13,-0.3,0.2,0.9,0.0,-0.2,0.0,0.0,0.0,0.0,0.0,0.0,-2.2
2013/14,0.5,1.6,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,-4.5
2014/15,-0.4,0.8,0.8,0.0,-0.1,-0.1,0.0,0.0,0.0,0.0,0.0,-0.2
2015/16,0.7,0.1,-0.2,0.0,-0.5,0.0,0.0,0.0,0.0,0.0,0.0,3.1
2016/17,1.9,2.7,1.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-5.4
2017/18,0.1,-1.1,-0.6,0.0,0.7,0.0,0.0,0.0,0.0,0.0,0.0,0.5
2018/19,0.1,0.6,-0.1,0.0,0.2,0.0,0.0,NaN,NaN,0.0,0.0,1.0


In [178]:
# ============================================================
# CHANGE FROM MAY
# ============================================================

may_values = (
    mom[mom["Month"] == 5]
    [["Year", "Value"]]
    .rename(columns={"Value": "May Value"})
)

may_change = mom.merge(
    may_values,
    on="Year",
    how="left"
)

may_change["Change from May"] = (
    may_change["Value"]
    - may_change["May Value"]
)

In [179]:
# ============================================================
# MAY REVISION TABLE
# ============================================================

df_may_change = may_change.pivot_table(
    index="Marketing Year",
    columns="Month",
    values="Change from May"
)

# May → Jun → Jul → ... → Apr
month_order = [
    5, 6, 7, 8, 9, 10, 11, 12,
    1, 2, 3, 4
]

df_may_change = df_may_change.reindex(
    columns=month_order
)

df_may_change = df_may_change.rename(
    columns=month_names
)

In [180]:
may_value_series = (
    may_change
    .drop_duplicates("Marketing Year")
    .set_index("Marketing Year")["May Value"]
)

df_may_change["May Value"] = may_value_series

df_may_change = df_may_change[
    ["May Value"] +
    [col for col in df_may_change.columns if col != "May Value"]
]

df_may_change.index = (
    df_may_change.index.astype(int).astype(str)
    + "/"
    + (df_may_change.index.astype(int) + 1)
        .astype(str)
        .str[-2:]
)

df_may_change

#wheat yield against may yields absolute values

Month,May Value,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Jan,Feb,Mar,Apr
Marketing Year,,,,,,,,,,,,,
2009/10,43.4,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010/11,43.4,0.0,0.5,2.5,3.5,3.5,3.3,3.0,3.0,3.9,3.9,3.9,3.9
2011/12,42.5,0.0,0.6,2.1,2.7,2.7,1.4,1.2,1.2,-2.0,-2.0,-2.0,-2.0
2012/13,45.7,0.0,-0.3,-0.1,0.8,0.8,0.6,0.6,0.6,2.2,2.2,2.2,2.2
2013/14,44.1,0.0,0.5,2.1,2.1,2.1,NaN,3.1,3.1,4.5,4.5,4.5,4.5
2014/15,42.7,0.0,-0.4,0.4,1.2,1.2,1.1,1.0,1.0,0.2,0.2,0.2,0.2
2015/16,46.7,0.0,0.7,0.8,0.6,0.6,0.1,0.1,0.1,-3.1,-3.1,-3.1,-3.1
2016/17,46.7,0.0,1.9,4.6,5.9,5.9,5.9,5.9,5.9,5.4,5.4,5.4,5.4
2017/18,47.2,0.0,0.1,-1.0,-1.6,-1.6,-0.9,-0.9,-0.9,-0.5,-0.5,-0.5,-0.5


In [181]:
df_yoy.rename(
    columns = month_names
)


Month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Year,,,,,,,,,,,,
2011,NaN,NaN,NaN,NaN,-0.9,-0.8,-1.3,-1.7,-1.7,-2.8,-2.7,-2.7
2012,-2.7,-2.7,-2.7,-2.7,3.2,2.3,1.0,1.3,1.3,2.4,2.6,2.6
2013,2.6,2.6,2.6,2.6,-1.6,-0.8,0.6,-0.3,-0.3,NaN,0.9,0.9
2014,0.9,0.9,0.9,0.9,-1.4,-2.3,-3.1,-2.3,-2.3,NaN,-3.5,-3.5
2015,-3.5,-3.5,-3.5,-3.5,0.8,1.9,1.2,0.2,0.2,-0.2,-0.1,-0.1
2016,-0.1,-0.1,-0.1,-0.1,3.2,4.4,7.0,8.5,8.5,9.0,9.0,9.0
2017,9.0,9.0,9.0,9.0,0.5,-1.3,-5.1,-7.0,-7.0,-6.3,-6.3,-6.3
2018,-6.3,-6.3,-6.3,-6.3,-0.4,-0.4,1.3,1.8,1.8,1.3,1.3,1.3
2019,NaN,1.3,1.3,1.3,1.8,1.8,2.5,4.2,4.2,4.0,4.1,4.1


In [170]:
display(df_mom)

Month,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Jan,Feb,Mar,Apr,May
Marketing Year,,,,,,,,,,,,
2010/11,0.5,2.0,1.0,0.0,-0.2,-0.3,0.0,0.0,0.0,0.0,0.0,-3.9
2011/12,0.6,1.5,0.6,0.0,-1.3,-0.2,0.0,0.0,0.0,0.0,0.0,2.0
2012/13,-0.3,0.2,0.9,0.0,-0.2,0.0,0.0,0.0,0.0,0.0,0.0,-2.2
2013/14,0.5,1.6,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,-4.5
2014/15,-0.4,0.8,0.8,0.0,-0.1,-0.1,0.0,0.0,0.0,0.0,0.0,-0.2
2015/16,0.7,0.1,-0.2,0.0,-0.5,0.0,0.0,0.0,0.0,0.0,0.0,3.1
2016/17,1.9,2.7,1.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-5.4
2017/18,0.1,-1.1,-0.6,0.0,0.7,0.0,0.0,0.0,0.0,0.0,0.0,0.5
2018/19,0.1,0.6,-0.1,0.0,0.2,0.0,0.0,NaN,NaN,0.0,0.0,1.0


In [171]:
pivot

Month,January,February,March,April,May,June,July,August,September,October,November,December
Year,,,,,,,,,,,,
2010,NaN,NaN,NaN,NaN,43.4,43.9,45.9,46.9,46.9,46.7,46.4,46.4
2011,46.4,46.4,46.4,46.4,42.5,43.1,44.6,45.2,45.2,43.9,43.7,43.7
2012,43.7,43.7,43.7,43.7,45.7,45.4,45.6,46.5,46.5,46.3,46.3,46.3
2013,46.3,46.3,46.3,46.3,44.1,44.6,46.2,46.2,46.2,NaN,47.2,47.2
2014,47.2,47.2,47.2,47.2,42.7,42.3,43.1,43.9,43.9,43.8,43.7,43.7
2015,43.7,43.7,43.7,43.7,43.5,44.2,44.3,44.1,44.1,43.6,43.6,43.6
2016,43.6,43.6,43.6,43.6,46.7,48.6,51.3,52.6,52.6,52.6,52.6,52.6
2017,52.6,52.6,52.6,52.6,47.2,47.3,46.2,45.6,45.6,46.3,46.3,46.3
2018,46.3,46.3,46.3,46.3,46.8,46.9,47.5,47.4,47.4,47.6,47.6,47.6


In [190]:
mom

,Year,Month,Value,Marketing Year,Previous Month Value,MoM Change
0,2024,1,48.6,2023,48.6,0.0
1,2024,2,48.6,2023,48.6,0.0
2,2024,3,48.6,2023,48.6,0.0
3,2024,4,48.6,2023,48.6,0.0
4,2024,5,48.9,2023,48.6,0.3
...,...,...,...,...,...,...
189,2026,5,47.5,2025,53.3,-5.8
190,2026,6,47.0,2026,47.5,-0.5
191,2026,7,47.9,2026,47.0,0.9
192,2026,8,47.8,2026,47.9,-0.1


,Report Date,Day,Month,Year,Value,commodity,country,snd_item,wheat_class,crop,table,crop year,month offset
271,2024/1/12,12,1,2024,37.300000,wheat,united states,harvested area,NaN,nc,us,2023,1_01
272,2024/1/12,12,1,2024,49.600000,wheat,united states,planted area,NaN,nc,us,2023,1_01
273,2024/1/12,12,1,2024,570.000000,wheat,united states,beginning stocks,NaN,nc,us,2023,1_01
274,2024/1/12,12,1,2024,1154.000000,wheat,united states,domestic total,NaN,nc,us,2023,1_01
275,2024/1/12,12,1,2024,648.000000,wheat,united states,ending stocks,NaN,nc,us,2023,1_01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
200753,2026/9/11,11,9,2026,775.000000,wheat,united states,exports,NaN,nc,us,2026,0_09
200755,2026/9/11,11,9,2026,1874.000000,wheat,united states,use total,NaN,nc,us,2026,0_09
200757,2026/9/11,11,9,2026,717.000000,wheat,united states,ending stocks,NaN,nc,us,2026,0_09
200759,2026/9/11,11,9,2026,6.400000,wheat,united states,avg. farm price,NaN,nc,us,2026,0_09


,Report Date,Day,Month,Year,Value,commodity,country,snd_item,wheat_class,crop,table,crop year,month offset
191948,2026/5/12,12,5,2026,43.800000,wheat,united states,planted area,NaN,nc,us,2026,0_05
191950,2026/5/12,12,5,2026,32.900000,wheat,united states,harvested area,NaN,nc,us,2026,0_05
191952,2026/5/12,12,5,2026,47.500000,wheat,united states,yield,NaN,nc,us,2026,0_05
191954,2026/5/12,12,5,2026,935.000000,wheat,united states,beginning stocks,NaN,nc,us,2026,0_05
191956,2026/5/12,12,5,2026,1561.000000,wheat,united states,production,NaN,nc,us,2026,0_05
...,...,...,...,...,...,...,...,...,...,...,...,...,...
200753,2026/9/11,11,9,2026,775.000000,wheat,united states,exports,NaN,nc,us,2026,0_09
200755,2026/9/11,11,9,2026,1874.000000,wheat,united states,use total,NaN,nc,us,2026,0_09
200757,2026/9/11,11,9,2026,717.000000,wheat,united states,ending stocks,NaN,nc,us,2026,0_09
200759,2026/9/11,11,9,2026,6.400000,wheat,united states,avg. farm price,NaN,nc,us,2026,0_09


In [ ]:


filtered = df[
    (df["commodity"] == commodity) &
    (df["country"] == country) &
    (df["table"] == report) &
    (df["crop"] == crop)
].copy()

filtered = filtered[
    filtered["crop year"] == 2026
].copy()
filtered

filtered["Month"] = pd.to_datetime(
    filtered["Report Date"]
).dt.strftime("%b")

monthly_balance_sheet = filtered.pivot_table(
    index="snd_item",
    columns="Month",
    values="Value",
    aggfunc="last"
)

month_order = [
    "May", "Jun", "Jul", "Aug", "Sep", "Oct",
    "Nov", "Dec", "Jan", "Feb", "Mar", "Apr"
]

monthly_balance_sheet = monthly_balance_sheet.reindex(
    columns=month_order
)

monthly_balance_sheet

Month,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Jan,Feb,Mar,Apr
snd_item,,,,,,,,,,,,
avg. farm price,6.500000,6.000000,6.000000,6.200000,6.400000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
beginning stocks,935.000000,935.000000,920.000000,920.000000,920.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
domestic total,1099.000000,1099.000000,1099.000000,1099.000000,1099.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ending stocks,762.000000,744.000000,722.000000,717.000000,717.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
exports,775.000000,775.000000,775.000000,775.000000,775.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
feed residual,80.000000,80.000000,80.000000,80.000000,80.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
food,960.000000,960.000000,960.000000,960.000000,960.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
harvested area,32.900000,32.900000,32.100000,32.100000,32.100000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
imports,140.000000,140.000000,140.000000,140.000000,140.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [260]:
test = pd.read_csv(FILE_PATH)

def get_month_offset_order():
    """Return WASDE month offsets in marketing-year order."""
    return [
        f"{year}_{month:02d}"
        for year, months in [
            (0, range(5, 13)),
            (1, range(1, 13)),
            (2, range(1, 5)),
        ]
        for month in months
    ]

# Define the order
month_order = get_month_offset_order()

# Sort the DataFrame
test["month offset"] = pd.Categorical(
    test["month offset"].astype(str),
    categories=month_order,
    ordered=True,
)

test = test.sort_values("month offset")
filtering = test[
                (test["crop year"]== 2025)&
                (test["commodity"]== "corn")& 
                (test["snd_item"]== "production")&
                (test["table"]== "us")
                 ]
# Display the crop year and month offset in order
display(
    filtering[["crop year", "month offset", "Report Date", "Value","crop","snd_item"]]
)

,crop year,month offset,Report Date,Value,crop,snd_item
168014,2025,0_05,2025/5/12,15820.0,nc,production
170174,2025,0_06,2025/6/12,15820.0,nc,production
172334,2025,0_07,2025/7/11,15705.0,nc,production
174566,2025,0_08,2025/8/12,16742.0,nc,production
176797,2025,0_09,2025/9/12,16814.0,nc,production
179134,2025,0_11,2025/11/14,16752.0,nc,production
180862,2025,0_12,2025/12/9,16752.0,nc,production
183092,2025,1_01,2026/1/12,17021.0,nc,production
185322,2025,1_02,2026/2/10,17021.0,nc,production
187552,2025,1_03,2026/3/10,17021.0,nc,production


In [ ]:
"""
wasde_tables.py
Marketing-year pivot tables from the processed WASDE file.

  Table 1  build_yearly_table()   rows = snd_item (WASDE order), cols = marketing year (2025/26)
  Table 2  build_monthly_table()  rows = snd_item (WASDE order), cols = report months of ONE marketing year

How "month offset" works (from your data):
  "<year_off>_<calendar month>", where calendar year = crop year + year_off.
  e.g. crop year 2010, offset "1_10"  ->  Oct 2011.
  Each crop year therefore has up to 24 report months: 0_05 ... 2_04
    0_05 .. 1_04  -> 'nc' (projection for that marketing year)
    1_05 .. 2_04  -> 'oc' (old-crop estimate)
"""
from __future__ import annotations

import re

import pandas as pd

# ---------------------------------------------------------------------
# 1. CONFIG - WASDE display order per commodity
#    Names are matched after normalising (lower case, '&' -> 'and',
#    punctuation removed), so "Food, Seed & Industrial" == "food seed and industrial".
#    Adjust to your real snd_item values: print(df["snd_item"].unique())
#    Anything not listed is appended at the bottom (alphabetical), never dropped.
# ---------------------------------------------------------------------
MONTH_ABBR = ("Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec")

_AREA_YIELD = ["area planted", "area harvested", "yield"]
_SUPPLY = ["beginning stocks", "production", "imports", "supply total", "total supply"]

SND_ORDER_RAW = {
    "corn": _AREA_YIELD + _SUPPLY + [
        "feed and residual", "food seed and industrial", "ethanol and by-products",
        "domestic total", "exports", "use total", "total use", "ending stocks",
    ],
    "soybeans": _AREA_YIELD + _SUPPLY + [
        "crushings", "crush", "exports", "seed", "residual",
        "use total", "total use", "ending stocks",
    ],
    "wheat": _AREA_YIELD + _SUPPLY + [
        "food", "seed", "feed and residual",
        "domestic total", "exports", "use total", "total use", "ending stocks",
    ],
    "default": _AREA_YIELD + _SUPPLY + [
        "feed and residual", "food seed and industrial", "domestic total",
        "exports", "use total", "total use", "ending stocks",
    ],
}


def _norm(text) -> str:
    s = str(text).lower().replace("&", " and ")
    return re.sub(r"[^a-z0-9]+", " ", s).strip()


SND_RANK = {
    commodity: {_norm(name): i for i, name in enumerate(names)}
    for commodity, names in SND_ORDER_RAW.items()
}


def order_items(items, commodity: str | None = None) -> list:
    """Sort snd_item values in WASDE display order; unknown items go last."""
    rank = SND_RANK.get(_norm(commodity), SND_RANK["default"]) if commodity else SND_RANK["default"]
    big = len(rank) + 1
    return sorted(items, key=lambda x: (rank.get(_norm(x), big), str(x)))


# ---------------------------------------------------------------------
# 2. LABEL FORMATTERS
# ---------------------------------------------------------------------
def parse_month_offset(offset) -> tuple[int, int]:
    """'1_10' -> (1, 10)"""
    year_off, month = str(offset).split("_")
    return int(year_off), int(month)


def format_month_offset(offset, crop_year: int | None = None) -> str:
    """
    With crop_year : '1_10', 2010 -> 'Oct-11'
    Without        : '1_10'       -> 'Oct (Y1)'   (Y0 = start year of the marketing year)
    """
    year_off, month = parse_month_offset(offset)
    name = MONTH_ABBR[month - 1]
    if crop_year is None:
        return f"{name} (Y{year_off})"
    return f"{name}-{(int(crop_year) + year_off) % 100:02d}"


def format_marketing_year(crop_year, long: bool = False) -> str:
    """2025 -> '2025/26'  (long=True -> '2025/2026'); handles 1999 -> '1999/00'"""
    y = int(crop_year)
    return f"{y}/{y + 1}" if long else f"{y}/{(y + 1) % 100:02d}"


# ---------------------------------------------------------------------
# 3. PREPARE / FILTER
# ---------------------------------------------------------------------
def prepare(df: pd.DataFrame) -> pd.DataFrame:
    """Standardise column names/types once, right after reading the CSV."""
    d = df.copy()
    d.columns = [str(c).strip() for c in d.columns]
    d = d.rename(columns={
        "Report Date": "report_date", "Value": "value",
        "crop year": "crop_year", "month offset": "month_offset",
    })

    d["report_date"] = pd.to_datetime(d["report_date"], errors="coerce")
    d["value"] = pd.to_numeric(d["value"], errors="coerce")
    d["crop_year"] = pd.to_numeric(d["crop_year"], errors="coerce")

    for col in ("commodity", "country", "snd_item", "wheat_class", "crop", "table"):
        if col in d.columns:
            d[col] = d[col].astype("string").str.strip().str.lower()

    parts = d["month_offset"].astype("string").str.split("_", n=1, expand=True)
    d["year_off"] = pd.to_numeric(parts[0], errors="coerce")
    d["month_num"] = pd.to_numeric(parts[1], errors="coerce")

    d = d.dropna(subset=["report_date", "value", "crop_year", "year_off", "month_num"]).copy()
    for col in ("crop_year", "year_off", "month_num"):
        d[col] = d[col].astype(int)

    # canonical zero-padded key, e.g. "1_05"
    d["month_offset"] = d["year_off"].astype(str) + "_" + d["month_num"].astype(str).str.zfill(2)
    return d


def select(df, commodity, country, table=None, wheat_class=None) -> pd.DataFrame:
    """Narrow to ONE balance sheet so each (snd_item, period) is unique."""
    mask = (df["commodity"] == commodity.lower()).fillna(False)
    mask &= (df["country"] == country.lower()).fillna(False)
    if table:
        mask &= (df["table"] == table.lower()).fillna(False)
    if wheat_class:
        mask &= (df["wheat_class"] == wheat_class.lower()).fillna(False)
    return df[mask].copy()


def _latest(d: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    """If a key repeats, keep the row with the most recent report_date."""
    return d.sort_values("report_date").drop_duplicates(keys, keep="last")


def _commodity_of(d: pd.DataFrame):
    return d["commodity"].dropna().iloc[0] if not d["commodity"].dropna().empty else None


# ---------------------------------------------------------------------
# 4. TABLE 1 - YEARLY (rows: snd_item, cols: marketing year)
# ---------------------------------------------------------------------
def build_yearly_table(
    df: pd.DataFrame,
    as_of: str = "latest",
    first_year: int | None = None,
    last_year: int | None = None,
) -> pd.DataFrame:
    """
    as_of = "latest" -> most recent estimate available for each marketing year
    as_of = "1_05"   -> the value as published in that report month
                        (offset 1_05 = May report after the marketing year began);
                        years that haven't reached that report yet show NaN
    """
    d = df
    if as_of != "latest":
        y, m = parse_month_offset(as_of)
        d = d[(d["year_off"] == y) & (d["month_num"] == m)]
    if first_year is not None:
        d = d[d["crop_year"] >= first_year]
    if last_year is not None:
        d = d[d["crop_year"] <= last_year]
    if d.empty:
        return pd.DataFrame()

    commodity = _commodity_of(d)
    d = _latest(d, ["snd_item", "crop_year"])

    out = d.pivot(index="snd_item", columns="crop_year", values="value").sort_index(axis=1)
    out = out.reindex(order_items(out.index, commodity))
    out.columns = [format_marketing_year(c) for c in out.columns]
    out.index.name = "snd_item"
    out.columns.name = "marketing year"
    return out


# ---------------------------------------------------------------------
# 5. TABLE 2 - MONTHLY (rows: snd_item, cols: report months of one marketing year)
# ---------------------------------------------------------------------
def build_monthly_table(
    df: pd.DataFrame,
    crop_year: int,
    crop_status: str | None = None,
) -> pd.DataFrame:
    """
    crop_year   : marketing year start, e.g. 2024 -> 2024/25
    crop_status : None = all 24 report months, "nc" = projection months, "oc" = old-crop months
    """
    d = df[df["crop_year"] == int(crop_year)]
    if crop_status:
        d = d[d["crop"] == crop_status.lower()]
    if d.empty:
        return pd.DataFrame()

    commodity = _commodity_of(d)
    d = _latest(d, ["snd_item", "month_offset"])

    # chronological column order: (year_off, month)
    col_order = (
        d[["month_offset", "year_off", "month_num"]]
        .drop_duplicates()
        .sort_values(["year_off", "month_num"])["month_offset"]
        .tolist()
    )

    out = d.pivot(index="snd_item", columns="month_offset", values="value")[col_order]
    out = out.reindex(order_items(out.index, commodity))
    out.columns = [format_month_offset(o, crop_year) for o in col_order]
    out.index.name = "snd_item"
    out.columns.name = f"MY {format_marketing_year(crop_year)}"
    return out


# ---------------------------------------------------------------------
# 6. DEMO
# ---------------------------------------------------------------------
if __name__ == "__main__":
    raw = pd.read_csv(FILE_PATH)  # sep="\t" if tab-separated
    data = prepare(raw)

 

corn_us = select(
    data,
    commodity="soybe",
    country="united states",
    table="us",
)

yearly = build_yearly_table(corn_us)
monthly = build_monthly_table(
    corn_us,
    crop_year=2024,
    crop_status="nc",
)

display(yearly)
display(monthly)

marketing year,2009/10,2010/11,2011/12,2012/13,2013/14,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25,2025/26,2026/27
snd_item,,,,,,,,,,,,,,,,,,
yield,44.000000,43.500000,41.900000,39.800000,44.000000,47.500000,48.000000,52.000000,49.300000,50.60000,47.400000,51.00000,51.70000,49.600000,50.600000,50.700000,53.000000,52.80000
beginning stocks,138.000000,151.000000,215.000000,169.000000,141.000000,92.000000,191.000000,197.000000,302.000000,438.00000,909.000000,525.00000,257.00000,274.000000,264.000000,342.000000,325.000000,325.00000
production,3359.000000,3329.000000,3094.000000,3034.000000,3358.000000,3927.000000,3926.000000,4296.000000,4412.000000,4428.00000,3552.000000,4216.00000,4465.00000,4270.000000,4162.000000,4374.000000,4262.000000,4535.00000
imports,15.000000,14.000000,16.000000,36.000000,72.000000,33.000000,24.000000,22.000000,22.000000,14.00000,15.000000,20.00000,16.00000,25.000000,21.000000,29.000000,25.000000,25.00000
supply total,3512.000000,3495.000000,3325.000000,3239.000000,3570.000000,4052.000000,4140.000000,4515.000000,4735.000000,4880.00000,4476.000000,4761.00000,4738.00000,4569.000000,4447.000000,4746.000000,4612.000000,4885.00000
exports,1501.000000,1501.000000,1362.000000,1320.000000,1647.000000,1843.000000,1936.000000,2174.000000,2129.000000,1748.00000,1682.000000,2261.00000,2158.00000,1992.000000,1695.000000,1882.000000,1520.000000,1685.00000
seed,90.000000,87.000000,90.000000,89.000000,97.000000,96.000000,97.000000,105.000000,104.000000,88.00000,96.000000,101.00000,102.00000,97.000000,75.000000,70.000000,75.000000,73.00000
residual,18.000000,43.000000,1.000000,1.000000,0.000000,49.000000,25.000000,34.000000,9.000000,43.00000,9.000000,1.00000,1.00000,4.000000,49.000000,23.000000,37.000000,38.00000
use total,3361.000000,3280.000000,3155.000000,3099.000000,3478.000000,3862.000000,3944.000000,4213.000000,4297.000000,3971.00000,3952.000000,4504.00000,4464.00000,4305.000000,4105.000000,4421.000000,4287.000000,4575.00000


MY 2024/25,May-24,Jun-24,Jul-24,Aug-24,Sep-24,Oct-24,Nov-24,Dec-24,Jan-25,Feb-25,Mar-25,Apr-25
snd_item,,,,,,,,,,,,
yield,52.000000,52.000000,52.000000,53.200000,53.200000,53.100000,51.700000,51.700000,50.700000,50.700000,50.700000,50.700000
beginning stocks,340.000000,350.000000,345.000000,345.000000,340.000000,342.000000,342.000000,342.000000,342.000000,342.000000,342.000000,342.000000
production,4450.000000,4450.000000,4435.000000,4589.000000,4586.000000,4582.000000,4461.000000,4461.000000,4366.000000,4366.000000,4366.000000,4366.000000
imports,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,20.000000,20.000000,20.000000,25.000000
supply total,4805.000000,4815.000000,4795.000000,4949.000000,4941.000000,4939.000000,4818.000000,4818.000000,4729.000000,4729.000000,4729.000000,4734.000000
exports,1825.000000,1825.000000,1825.000000,1850.000000,1850.000000,1850.000000,1825.000000,1825.000000,1825.000000,1825.000000,1825.000000,1825.000000
seed,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,75.000000,72.000000
residual,32.000000,32.000000,32.000000,36.000000,38.000000,36.000000,35.000000,35.000000,36.000000,36.000000,39.000000,42.000000
use total,4360.000000,4360.000000,4360.000000,4389.000000,4391.000000,4389.000000,4348.000000,4348.000000,4349.000000,4349.000000,4349.000000,4359.000000


In [ ]:

import pandas as pd


# 1. WASDE DISPLAY ORDER

MONTH_ABBR = (
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
)

AREA_YIELD_ITEMS = [
    "area planted",
    "area harvested",
    "yield",
]

SUPPLY_ITEMS = [
    "beginning stocks",
    "production",
    "imports",
    "supply total",
    "total supply",
]

WASDE_ITEM_ORDER = {
    "corn": AREA_YIELD_ITEMS + SUPPLY_ITEMS + [
        "feed and residual",
        "food seed and industrial",
        "ethanol and by-products",
        "domestic total",
        "exports",
        "use total",
        "total use",
        "ending stocks",
    ],
    "soybeans": AREA_YIELD_ITEMS + SUPPLY_ITEMS + [
        "crushings",
        "crush",
        "exports",
        "seed",
        "residual",
        "use total",
        "total use",
        "ending stocks",
    ],
    "wheat": AREA_YIELD_ITEMS + SUPPLY_ITEMS + [
        "food",
        "seed",
        "feed and residual",
        "domestic total",
        "exports",
        "use total",
        "total use",
        "ending stocks",
    ],
    "default": AREA_YIELD_ITEMS + SUPPLY_ITEMS + [
        "feed and residual",
        "food seed and industrial",
        "domestic total",
        "exports",
        "use total",
        "total use",
        "ending stocks",
    ],
}


# WASDE DISPLAY NAMES

WASDE_ITEM_LABELS = {
    "area planted": "Area Planted",
    "area harvested": "Area Harvested",
    "yield": "Yield",
    "beginning stocks": "Beginning Stocks",
    "production": "Production",
    "imports": "Imports",
    "supply total": "Total Supply",
    "total supply": "Total Supply",
    "crushings": "Crushings",
    "crush": "Crush",
    "feed and residual": "Feed and Residual",
    "food seed and industrial": "Food, Seed & Industrial",
    "ethanol and by-products": "Ethanol & By-products",
    "food": "Food",
    "seed": "Seed",
    "residual": "Residual",
    "domestic total": "Domestic Use",
    "exports": "Exports",
    "use total": "Total Use",
    "total use": "Total Use",
    "ending stocks": "Ending Stocks",
}


def format_wasde_item(item_name):
    """Convert internal WASDE item names to display labels."""
    return WASDE_ITEM_LABELS.get(
        item_name,
        item_name.replace("_", " ").title(),
    )


# 2. LABEL FORMATTERS

def order_wasde_items(item_names, commodity_name=None):
    """Sort WASDE items into the configured display order."""
    item_order = WASDE_ITEM_ORDER.get(
        commodity_name,
        WASDE_ITEM_ORDER["default"],
    )

    item_ranks = {
        item_name: position
        for position, item_name in enumerate(item_order)
    }

    return sorted(
        item_names,
        key=lambda item_name: (
            item_ranks.get(item_name, len(item_ranks)),
            item_name,
        ),
    )


def parse_month_offset(month_offset):
    """Convert '1_10' into (1, 10)."""
    year_offset, calendar_month = str(month_offset).split("_")
    return int(year_offset), int(calendar_month)


def format_month_offset(month_offset, crop_year=None):
    """
    With crop_year: '1_10', 2010 -> 'Oct-11'
    Without crop_year: '1_10' -> 'Oct (Y1)'
    """
    year_offset, calendar_month = parse_month_offset(month_offset)
    month_name = MONTH_ABBR[calendar_month - 1]

    if crop_year is None:
        return f"{month_name} (Y{year_offset})"

    calendar_year = int(crop_year) + year_offset
    return f"{month_name}-{calendar_year % 100:02d}"


def format_marketing_year(crop_year, show_full_year=False):
    """Convert 2025 to '2025/26' or '2025/2026'."""
    crop_year = int(crop_year)
    next_year = crop_year + 1

    if show_full_year:
        return f"{crop_year}/{next_year}"

    return f"{crop_year}/{next_year % 100:02d}"


# 3. PREPARE AND FILTER DATA

def prepare_wasde_data(dataframe):
    """Standardize WASDE column names and data types."""
    prepared_data = dataframe.copy()
    prepared_data.columns = [
        str(column_name).strip()
        for column_name in prepared_data.columns
    ]

    prepared_data = prepared_data.rename(columns={
        "Report Date": "report_date",
        "Value": "value",
        "crop year": "crop_year",
        "month offset": "month_offset",
    })

    prepared_data["report_date"] = pd.to_datetime(
        prepared_data["report_date"],
        errors="coerce",
    )
    prepared_data["value"] = pd.to_numeric(
        prepared_data["value"],
        errors="coerce",
    )
    prepared_data["crop_year"] = pd.to_numeric(
        prepared_data["crop_year"],
        errors="coerce",
    )

    text_columns = [
        "commodity",
        "country",
        "snd_item",
        "wheat_class",
        "crop",
        "table",
    ]

    for column_name in text_columns:
        if column_name in prepared_data.columns:
            prepared_data[column_name] = (
                prepared_data[column_name]
                .astype("string")
                .str.strip()
                .str.lower()
            )

    month_offset_parts = (
        prepared_data["month_offset"]
        .astype("string")
        .str.split("_", n=1, expand=True)
    )

    prepared_data["year_off"] = pd.to_numeric(
        month_offset_parts[0],
        errors="coerce",
    )
    prepared_data["month_num"] = pd.to_numeric(
        month_offset_parts[1],
        errors="coerce",
    )

    prepared_data = prepared_data.dropna(
        subset=[
            "report_date",
            "value",
            "crop_year",
            "year_off",
            "month_num",
        ]
    ).copy()

    for column_name in ["crop_year", "year_off", "month_num"]:
        prepared_data[column_name] = (
            prepared_data[column_name].astype(int)
        )

    prepared_data["month_offset"] = (
        prepared_data["year_off"].astype(str)
        + "_"
        + prepared_data["month_num"].astype(str).str.zfill(2)
    )

    return prepared_data


def select_wasde_data(
    dataframe,
    commodity,
    country,
    table=None,
    wheat_class=None,
):
    """Filter to one commodity, country and optional table or wheat class."""
    selected_data = dataframe[
        dataframe["commodity"].eq(commodity.lower()).fillna(False)
        & dataframe["country"].eq(country.lower()).fillna(False)
    ].copy()

    if table:
        selected_data = selected_data[
            selected_data["table"].eq(table.lower()).fillna(False)
        ]

    if wheat_class:
        selected_data = selected_data[
            selected_data["wheat_class"]
            .eq(wheat_class.lower())
            .fillna(False)
        ]

    return selected_data


# 4. SHARED TABLE HELPERS

def get_latest_records(dataframe, key_columns):
    """Keep the latest report for each combination of key columns."""
    return (
        dataframe.sort_values("report_date")
        .drop_duplicates(subset=key_columns, keep="last")
    )


def get_commodity_name(dataframe):
    """Return the commodity name for the selected data."""
    commodity_names = dataframe["commodity"].dropna()

    if commodity_names.empty:
        return None

    return commodity_names.iloc[0]


def order_balance_sheet_rows(balance_sheet, commodity_name=None):
    """Apply the configured WASDE item order."""
    ordered_item_names = order_wasde_items(
        balance_sheet.index,
        commodity_name,
    )

    return balance_sheet.reindex(ordered_item_names)


# 5. TABLE 1: YEARLY BALANCE SHEET

def build_yearly_table(
    dataframe,
    as_of="latest",
    first_year=None,
    last_year=None,
):
    """
    Rows: WASDE items.
    Columns: marketing years, e.g. 2025/26.

    as_of='latest': latest estimate for each item and crop year.
    as_of='1_05': reports with month offset 1_05 only.
    """
    yearly_data = dataframe.copy()

    if as_of != "latest":
        year_offset, calendar_month = parse_month_offset(as_of)

        yearly_data = yearly_data[
            yearly_data["year_off"].eq(year_offset)
            & yearly_data["month_num"].eq(calendar_month)
        ]

    if first_year is not None:
        yearly_data = yearly_data[
            yearly_data["crop_year"] >= first_year
        ]

    if last_year is not None:
        yearly_data = yearly_data[
            yearly_data["crop_year"] <= last_year
        ]

    if yearly_data.empty:
        return pd.DataFrame()

    commodity_name = get_commodity_name(yearly_data)

    latest_records = get_latest_records(
        yearly_data,
        ["snd_item", "crop_year"],
    )

    balance_sheet = latest_records.pivot(
        index="snd_item",
        columns="crop_year",
        values="value",
    )

    balance_sheet = order_balance_sheet_rows(
        balance_sheet,
        commodity_name,
    )

    balance_sheet.columns = [
        format_marketing_year(crop_year)
        for crop_year in balance_sheet.columns
    ]

    balance_sheet.index = [
        format_wasde_item(item_name)
        for item_name in balance_sheet.index
        ]
    balance_sheet.index.name = "WASDE Item"
    balance_sheet.columns.name = "Marketing Year"

    return balance_sheet


# 6. TABLE 2: MONTHLY BALANCE SHEET

def build_monthly_table(
    dataframe,
    crop_year,
    crop_status=None,
):
    """
    Rows: WASDE items.
    Columns: report months for one crop year.
    """
    monthly_data = dataframe[
        dataframe["crop_year"].eq(int(crop_year))
    ].copy()

    if crop_status:
        monthly_data = monthly_data[
            monthly_data["crop"].eq(crop_status.lower())
        ]

    if monthly_data.empty:
        return pd.DataFrame()

    commodity_name = get_commodity_name(monthly_data)

    latest_records = get_latest_records(
        monthly_data,
        ["snd_item", "month_offset"],
    )

    month_order = (
        latest_records[
            ["month_offset", "year_off", "month_num"]
        ]
        .drop_duplicates()
        .sort_values(["year_off", "month_num"])
        ["month_offset"]
        .tolist()
    )

    balance_sheet = latest_records.pivot(
        index="snd_item",
        columns="month_offset",
        values="value",
    )

    balance_sheet = balance_sheet.reindex(columns=month_order)
    balance_sheet = order_balance_sheet_rows(
        balance_sheet,
        commodity_name,
    )

    balance_sheet.columns = [
        format_month_offset(month_offset, crop_year)
        for month_offset in month_order
    ]

    balance_sheet.index = [
    format_wasde_item(item_name)
    for item_name in balance_sheet.index
    ]
    balance_sheet.index.name = "WASDE Item"
    balance_sheet.columns.name = (
        f"Marketing Year {format_marketing_year(crop_year)}"
    )

    return balance_sheet


,Report Date,snd_item,crop year,month offset,Value
192031,2026/5/12,avg. farm price,2025,NaN,4.15
194191,2026/6/11,avg. farm price,2025,NaN,4.15
196351,2026/7/10,avg. farm price,2025,NaN,4.15
198583,2026/8/12,avg. farm price,2025,NaN,4.15
200814,2026/9/11,avg. farm price,2025,NaN,4.15
192009,2026/5/12,beginning stocks,2025,NaN,1551.00
194169,2026/6/11,beginning stocks,2025,NaN,1551.00
196329,2026/7/10,beginning stocks,2025,NaN,1551.00
198561,2026/8/12,beginning stocks,2025,NaN,1551.00
200792,2026/9/11,beginning stocks,2025,NaN,1551.00


In [ ]:

import pandas as pd


# ============================================================
# 1. WASDE DISPLAY ORDER
# ============================================================

MONTH_ABBR = (
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
)


AREA_YIELD_ITEMS = [
    "area planted",
    "planted area",
    "planted",
    "area harvested",
    "harvested area",
    "harvested",
    "yield",
]

SUPPLY_ITEMS = [
    "beginning stocks",
    "production",
    "imports",
    "supply total",
    "total supply",
]

WASDE_ITEM_ORDER = {
    "corn": AREA_YIELD_ITEMS + SUPPLY_ITEMS + [
        "domestic feed or crush",
        "feed residual",
        "food seed industrial",
        "food feed industrial",
        "ethanol",
        "crushing",
        "crushings",
        "crush",
        "bio diesel",
        "seed",
        "food",
        "residual",
        "domestic total",
        "total domestic",
        "domestic use",
        "domestic disappearance",
        "exports",
        "exports total",
        "use total",
        "unaccounted",
        "loss",
        "total use",
        "ending stocks",
        "stock to use",
        "farm price",
        "avg. farm price",
        "avg. price",
        "average farm price",
        "season-average farm price",
    ],

    "soybeans": AREA_YIELD_ITEMS + SUPPLY_ITEMS + [
        "crushing",
        "crushings",
        "crush",
        "domestic feed or crush",
        "food seed industrial",
        "food feed industrial",
        "seed",
        "residual",
        "bio diesel",
        "domestic total",
        "total domestic",
        "domestic use",
        "domestic disappearance",
        "exports",
        "exports total",
        "use total",
        "unaccounted",
        "loss",
        "total use",
        "ending stocks",
        "stock to use",
        "avg. farm price",
        "avg. price",
    ],

    "wheat": AREA_YIELD_ITEMS + SUPPLY_ITEMS + [
        "food",
        "seed",
        "feed residual",
        "residual",
        "domestic total",
        "total domestic",
        "domestic use",
        "domestic disappearance",
        "exports",
        "exports total",
        "use total",
        "unaccounted",
        "loss",
        "total use",
        "ending stocks",
        "stock to use",
        "avg. farm price",
        "avg. price",
    ],

    "default": AREA_YIELD_ITEMS + SUPPLY_ITEMS + [
        "feed residual",
        "food seed industrial",
        "food feed industrial",
        "domestic total",
        "total domestic",
        "domestic use",
        "domestic disappearance",
        "exports",
        "exports total",
        "use total",
        "unaccounted",
        "loss",
        "total use",
        "ending stocks",
        "stock to use",
        "avg. farm price",
        "avg. price",
    ],
}



# ============================================================
# 2. WASDE DISPLAY LABELS
# ============================================================


WASDE_ITEM_LABELS = {
    # Area and yield
    "area planted": "Area Planted",
    "planted area": "Area Planted",
    "planted": "Area Planted",
    "area harvested": "Area Harvested",
    "harvested area": "Area Harvested",
    "harvested": "Area Harvested",
    "yield": "Yield",

    # Supply
    "beginning stocks": "Beginning Stocks",
    "production": "Production",
    "imports": "Imports",
    "supply total": "Total Supply",
    "total supply": "Total Supply",

    # Domestic use
    "domestic feed or crush": "Domestic Feed or Crush",
    "feed and residual": "Feed & Residual",
    "feed residual": "Feed & Residual",
    "food seed industrial": "Food, Seed & Industrial",
    "food seed and industrial": "Food, Seed & Industrial",
    "food feed industrial": "Food, Feed & Industrial",
    "ethanol": "Ethanol",
    "ethanol and by-products": "Ethanol & By-products",
    "crushing": "Crushing",
    "crushings": "Crushings",
    "crush": "Crush",
    "bio diesel": "Biodiesel",
    "food": "Food",
    "seed": "Seed",
    "residual": "Residual",
    "domestic total": "Domestic Use",
    "total domestic": "Domestic Use",
    "domestic use": "Domestic Use",
    "domestic disappearance": "Domestic Disappearance",

    # Exports and total use
    "exports": "Exports",
    "exports total": "Total Exports",
    "use total": "Total Use",
    "total use": "Total Use",
    "unaccounted": "Unaccounted",
    "loss": "Loss",

    # Ending stocks and prices
    "ending stocks": "Ending Stocks",
    "stock to use": "Stocks-to-Use",
    "farm price": "Avg. Farm Price",
    "avg. farm price": "Avg. Farm Price",
    "avg. price": "Avg. Farm Price",
    "average farm price": "Avg. Farm Price",
    "season-average farm price": "Avg. Farm Price",
}



# ============================================================
# 8. GROUP WASDE ROWS INTO FUNDAMENTAL SECTIONS
# ============================================================


WASDE_SECTIONS = {
    "Area & Yield": [
        "Area Planted",
        "Area Harvested",
        "Yield",
    ],

    "Supply": [
        "Beginning Stocks",
        "Production",
        "Imports",
        "Total Supply",
    ],

    "Domestic Use": [
        "Domestic Feed or Crush",
        "Feed & Residual",
        "Food, Seed & Industrial",
        "Food, Feed & Industrial",
        "Food",
        "Seed",
        "Residual",
        "Ethanol",
        "Ethanol & By-products",
        "Crushing",
        "Crushings",
        "Crush",
        "Biodiesel",
        "Domestic Use",
        "Domestic Disappearance",
        "Unaccounted",
        "Loss",
    ],

    "Exports": [
        "Exports",
        "Total Exports",
    ],

    "Total Use & Ending Stocks": [
        "Total Use",
        "Ending Stocks",
        "Stocks-to-Use",
        "Avg. Farm Price",
    ],
}



def split_wasde_table(table):
    """Split a formatted WASDE table into ordered sections."""
    sections = {}
    assigned_positions = set()

    for section_name, item_labels in WASDE_SECTIONS.items():
        positions = [
            i
            for i, label in enumerate(table.index)
            if label in item_labels
        ]

        if positions:
            sections[section_name] = table.iloc[positions].copy()
            assigned_positions.update(positions)

    # Keep any unrecognized items instead of dropping them.
    other_positions = [
        i
        for i in range(len(table))
        if i not in assigned_positions
    ]

    if other_positions:
        sections["Other Items"] = table.iloc[other_positions].copy()

    return sections

def format_wasde_item(item_name):
    """Convert raw WASDE item names to display labels."""
    item_name = str(item_name).strip().lower()

    return WASDE_ITEM_LABELS.get(
        item_name,
        item_name.replace("_", " ").title(),
    )


# ============================================================
# 3. ORDERING AND LABEL FORMATTERS
# ============================================================

def order_wasde_items(item_names, commodity_name=None):
    """Apply configured order and preserve unknown-item order."""
    commodity_name = (
        str(commodity_name).strip().lower()
        if commodity_name is not None
        else None
    )

    item_order = WASDE_ITEM_ORDER.get(
        commodity_name,
        WASDE_ITEM_ORDER["default"],
    )

    item_ranks = {
        item_name: position
        for position, item_name in enumerate(item_order)
    }

    # Stable sorting keeps unknown items in their original order.
    return sorted(
        item_names,
        key=lambda item_name: item_ranks.get(
            str(item_name).strip().lower(),
            len(item_ranks),
        ),
    )


def parse_month_offset(month_offset):
    """Convert '1_10' into (1, 10)."""
    year_offset, calendar_month = str(month_offset).split("_")
    return int(year_offset), int(calendar_month)


def format_month_offset(month_offset, crop_year=None):
    """Format a month offset as 'Oct-26' or 'Oct (Y1)'."""
    year_offset, calendar_month = parse_month_offset(month_offset)
    month_name = MONTH_ABBR[calendar_month - 1]

    if crop_year is None:
        return f"{month_name} (Y{year_offset})"

    calendar_year = int(crop_year) + year_offset
    return f"{month_name}-{calendar_year % 100:02d}"


def format_marketing_year(crop_year, show_full_year=False):
    """Convert 2025 to '2025/26' or '2025/2026'."""
    crop_year = int(crop_year)
    next_year = crop_year + 1

    if show_full_year:
        return f"{crop_year}/{next_year}"

    return f"{crop_year}/{next_year % 100:02d}"


# ============================================================
# 4. PREPARE AND FILTER DATA
# ============================================================

def prepare_wasde_data(dataframe):
    """Standardize WASDE column names and data types."""
    prepared_data = dataframe.copy()

    prepared_data.columns = [
        str(column_name).strip()
        for column_name in prepared_data.columns
    ]

    prepared_data = prepared_data.rename(columns={
        "Report Date": "report_date",
        "Value": "value",
        "crop year": "crop_year",
        "month offset": "month_offset",
    })

    required_columns = [
        "report_date", "value", "crop_year", "month_offset",
        "commodity", "country", "snd_item",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in prepared_data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required WASDE columns: {missing_columns}"
        )

    prepared_data["report_date"] = pd.to_datetime(
        prepared_data["report_date"],
        errors="coerce",
    )

    prepared_data["value"] = pd.to_numeric(
        prepared_data["value"],
        errors="coerce",
    )

    prepared_data["crop_year"] = pd.to_numeric(
        prepared_data["crop_year"],
        errors="coerce",
    )

    text_columns = [
        "commodity",
        "country",
        "snd_item",
        "wheat_class",
        "crop",
        "table",
    ]

    for column_name in text_columns:
        if column_name in prepared_data.columns:
            prepared_data[column_name] = (
                prepared_data[column_name]
                .astype("string")
                .str.strip()
                .str.lower()
            )

    month_offset_parts = (
        prepared_data["month_offset"]
        .astype("string")
        .str.split("_", n=1, expand=True)
    )

    prepared_data["year_off"] = pd.to_numeric(
        month_offset_parts[0],
        errors="coerce",
    )

    prepared_data["month_num"] = pd.to_numeric(
        month_offset_parts[1],
        errors="coerce",
    )

    prepared_data = prepared_data.dropna(
        subset=[
            "report_date",
            "value",
            "crop_year",
            "year_off",
            "month_num",
        ]
    ).copy()

    for column_name in ["crop_year", "year_off", "month_num"]:
        prepared_data[column_name] = (
            prepared_data[column_name].astype(int)
        )

    prepared_data["month_offset"] = (
        prepared_data["year_off"].astype(str)
        + "_"
        + prepared_data["month_num"].astype(str).str.zfill(2)
    )

    return prepared_data


def select_wasde_data(
    dataframe,
    commodity,
    country,
    table=None,
    wheat_class=None,
):
    """Filter by commodity, country and optional table/class."""
    selected_data = dataframe[
        dataframe["commodity"].eq(commodity.lower()).fillna(False)
        & dataframe["country"].eq(country.lower()).fillna(False)
    ].copy()

    if table and "table" in selected_data.columns:
        selected_data = selected_data[
            selected_data["table"].eq(table.lower()).fillna(False)
        ]

    if wheat_class and "wheat_class" in selected_data.columns:
        selected_data = selected_data[
            selected_data["wheat_class"]
            .eq(wheat_class.lower())
            .fillna(False)
        ]

    return selected_data


# ============================================================
# 5. SHARED TABLE HELPERS
# ============================================================

def get_latest_records(dataframe, key_columns):
    """Keep the latest report for each combination of keys."""
    return (
        dataframe.sort_values("report_date")
        .drop_duplicates(subset=key_columns, keep="last")
    )


def get_commodity_name(dataframe):
    """Return the selected commodity name."""
    commodity_names = dataframe["commodity"].dropna()

    if commodity_names.empty:
        return None

    return str(commodity_names.iloc[0]).strip().lower()


def order_balance_sheet_rows(balance_sheet, commodity_name=None):
    """Order raw item names before applying display labels."""
    ordered_item_names = order_wasde_items(
        balance_sheet.index.tolist(),
        commodity_name,
    )

    return balance_sheet.reindex(ordered_item_names)


def style_wasde_table(table):
    """Display numeric values to 2 decimal places."""
    return table.style.format("{:.2f}", na_rep="—")


# ============================================================
# 6. TABLE 1: YEARLY BALANCE SHEET
# ============================================================

def build_yearly_table(
    dataframe,
    as_of="latest",
    first_year=None,
    last_year=None,
):
    """Rows are WASDE items; columns are marketing years."""
    yearly_data = dataframe.copy()

    if as_of != "latest":
        year_offset, calendar_month = parse_month_offset(as_of)

        yearly_data = yearly_data[
            yearly_data["year_off"].eq(year_offset)
            & yearly_data["month_num"].eq(calendar_month)
        ]

    if first_year is not None:
        yearly_data = yearly_data[
            yearly_data["crop_year"] >= first_year
        ]

    if last_year is not None:
        yearly_data = yearly_data[
            yearly_data["crop_year"] <= last_year
        ]

    if yearly_data.empty:
        return pd.DataFrame()

    commodity_name = get_commodity_name(yearly_data)

    latest_records = get_latest_records(
        yearly_data,
        ["snd_item", "crop_year"],
    )

    balance_sheet = latest_records.pivot(
        index="snd_item",
        columns="crop_year",
        values="value",
    )

    # Sort using raw WASDE names first.
    balance_sheet = order_balance_sheet_rows(
        balance_sheet,
        commodity_name,
    )

    # Sort years chronologically before formatting their labels.
    balance_sheet = balance_sheet.reindex(
        sorted(balance_sheet.columns),
        axis=1,
    )

    balance_sheet.columns = [
        format_marketing_year(crop_year)
        for crop_year in balance_sheet.columns
    ]

    # Apply display labels only after sorting.
    balance_sheet.index = [
        format_wasde_item(item_name)
        for item_name in balance_sheet.index
    ]

    balance_sheet.index.name = "WASDE Item"
    balance_sheet.columns.name = "Marketing Year"

    return balance_sheet


# ============================================================
# 7. TABLE 2: MONTHLY BALANCE SHEET
# ============================================================

def build_monthly_table(
    dataframe,
    crop_year,
    crop_status=None,
):
    """Rows are WASDE items; columns are report months."""
    monthly_data = dataframe[
        dataframe["crop_year"].eq(int(crop_year))
    ].copy()

    if crop_status and "crop" in monthly_data.columns:
        monthly_data = monthly_data[
            monthly_data["crop"].eq(crop_status.lower())
        ]

    if monthly_data.empty:
        return pd.DataFrame()

    commodity_name = get_commodity_name(monthly_data)

    latest_records = get_latest_records(
        monthly_data,
        ["snd_item", "month_offset"],
    )

    month_order = (
        latest_records[
            ["month_offset", "year_off", "month_num"]
        ]
        .drop_duplicates()
        .sort_values(["year_off", "month_num"])
        ["month_offset"]
        .tolist()
    )

    balance_sheet = latest_records.pivot(
        index="snd_item",
        columns="month_offset",
        values="value",
    )

    balance_sheet = balance_sheet.reindex(columns=month_order)

    # Sort using raw WASDE names first.
    balance_sheet = order_balance_sheet_rows(
        balance_sheet,
        commodity_name,
    )

    # Format month headers after chronological ordering.
    balance_sheet.columns = [
        format_month_offset(month_offset, crop_year)
        for month_offset in month_order
    ]

    # Apply display labels after row ordering.
    balance_sheet.index = [
        format_wasde_item(item_name)
        for item_name in balance_sheet.index
    ]

    balance_sheet.index.name = "WASDE Item"
    balance_sheet.columns.name = (
        f"Marketing Year {format_marketing_year(crop_year)}"
    )

    return balance_sheet


In [314]:

wasde = prepare_wasde_data(df)

corn_us = select_wasde_data(
    wasde,
    commodity="corn",
    country="united states",
)

# Inspect raw item names
print(sorted(corn_us["snd_item"].dropna().unique()))


# ============================================================
# 1. YEARLY BALANCE SHEET — GROUPED BY SECTION
# ============================================================

yearly_table = build_yearly_table(corn_us)

yearly_sections = split_wasde_table(yearly_table)

print("YEARLY BALANCE SHEET")

for section_name, section_table in yearly_sections.items():
    print(f"\n{section_name}")
    display(style_wasde_table(section_table))


# ============================================================
# 2. MONTHLY BALANCE SHEET — GROUPED BY SECTION
# ============================================================

monthly_table = build_monthly_table(
    corn_us,
    crop_year=2026,
)

monthly_sections = split_wasde_table(monthly_table)

print("\nMONTHLY BALANCE SHEET — 2026 CROP YEAR")

for section_name, section_table in monthly_sections.items():
    print(f"\n{section_name}")
    display(style_wasde_table(section_table))


['avg. farm price', 'beginning stocks', 'domestic feed or crush', 'domestic total', 'ending stocks', 'ethanol', 'exports', 'feed residual', 'food seed industrial', 'harvested area', 'imports', 'planted area', 'production', 'stock to use', 'supply total', 'use total', 'yield']
YEARLY BALANCE SHEET

Area & Yield


Marketing Year,2009/10,2010/11,2011/12,2012/13,2013/14,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25,2025/26,2026/27
WASDE Item,,,,,,,,,,,,,,,,,,
Area Planted,86.40,88.20,91.90,97.20,95.40,90.60,88.00,94.00,90.20,88.90,89.70,90.70,93.30,88.20,94.60,90.90,98.80,96.80
Area Harvested,79.50,81.40,84.00,87.40,87.50,83.10,80.80,86.70,82.70,81.30,81.30,82.30,85.30,78.70,86.50,83.00,91.30,88.50
Yield,164.70,152.80,147.20,123.40,158.10,171.00,168.40,174.60,176.60,176.40,167.50,171.40,176.70,173.40,177.30,179.30,186.50,178.50



Supply


Marketing Year,2009/10,2010/11,2011/12,2012/13,2013/14,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25,2025/26,2026/27
WASDE Item,,,,,,,,,,,,,,,,,,
Beginning Stocks,1673.00,1708.00,28.64,989.00,821.00,1232.00,1731.00,1737.00,2293.00,2140.00,2221.00,1919.00,1235.00,1377.00,1360.00,1763.00,1551.00,1922.00
Production,13092.00,316.17,12360.00,10780.00,13829.00,14216.00,13602.00,15148.00,14609.00,14340.00,13620.00,14111.00,15074.00,13651.00,15341.00,14892.00,17021.00,15800.00
Imports,8.00,28.00,29.00,162.00,36.00,32.00,67.00,57.00,36.00,28.00,42.00,24.00,24.00,39.00,28.00,22.00,30.00,25.00
Total Supply,14774.00,14182.00,13516.00,11932.00,14686.00,15479.00,15401.00,16942.00,16939.00,16509.00,15883.00,16055.00,16333.00,15066.00,16729.00,16677.00,18602.00,17747.00



Domestic Use


Marketing Year,2009/10,2010/11,2011/12,2012/13,2013/14,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25,2025/26,2026/27
WASDE Item,,,,,,,,,,,,,,,,,,
Feed & Residual,5140.00,4793.00,4545.00,4333.00,5036.00,5324.00,5120.00,5472.00,5304.00,5430.00,5897.00,5598.00,5721.00,5486.00,5805.00,5454.00,6350.00,5950.00
"Food, Seed & Industrial",5939.00,6428.00,6439.00,6046.00,6501.00,6560.00,6646.00,6883.00,7056.00,6793.00,6287.00,6470.00,6764.00,6558.00,6868.00,6813.00,6905.00,6955.00
Ethanol,4568.00,5021.00,5011.00,4648.00,5134.00,5200.00,5224.00,5432.00,5605.00,5378.00,4857.00,5033.00,5326.00,5176.00,5478.00,5436.00,5550.00,5600.00
Domestic Use,11079.00,11220.00,279.02,10379.00,11537.00,11883.00,11766.00,12356.00,12360.00,12223.00,12185.00,12068.00,12484.00,12045.00,12673.00,12267.00,13255.00,12905.00


... (14 outputs left out)


Total Use & Ending Stocks


Marketing Year 2026/27,May-26,Jun-26,Jul-26,Aug-26,Sep-26
WASDE Item,,,,,
Total Use,16205.00,16205.00,16255.00,16330.00,16180.00
Ending Stocks,1957.00,1960.00,1790.00,41.98,39.79
Avg. Farm Price,4.40,4.40,4.40,4.50,4.80



Other Items


Marketing Year 2026/27,May-26,Jun-26,Jul-26,Aug-26,Sep-26
WASDE Item,,,,,
Domestic Feed or Crush,154.95,154.95,154.95,154.95,151.14
Stock To Use,0.12,0.12,0.11,0.10,0.10


In [315]:
df["snd_item"].unique()

<ArrowStringArray>
[      'beginning stocks',           'domestic use',          'ending stocks',
                'exports',          'feed residual',                   'food',
                'imports',             'production',                   'seed',
           'supply total',              'use total',           'stock to use',
         'harvested area',           'planted area',         'domestic total',
                'ethanol',   'food seed industrial',                  'yield',
             'bio diesel',   'food feed industrial',               'crushing',
               'residual', 'domestic feed or crush',        'avg. farm price',
         'total domestic', 'domestic disappearance',             'avg. price',
                'planted',              'harvested',          'exports total',
            'unaccounted',                   'loss']
Length: 32, dtype: str

In [321]:


df = pd.read_csv(FILE_PATH)

# Keep only unique item combinations
audit = (
    df[["commodity", "country", "table", "snd_item"]]
    .drop_duplicates()
    .sort_values(["commodity", "country", "table", "snd_item"])
)

print("Unique combinations:", len(audit))
print("Unique commodities:", df["commodity"].nunique())
print("Unique raw item names:", df["snd_item"].nunique())

# Save a compact audit file
audit.to_csv("wasde_item_audit.csv", index=False)


Unique combinations: 1073
Unique commodities: 10
Unique raw item names: 32


In [320]:

import re
import pandas as pd

# Only add mappings after verifying their meaning.
WASDE_MAP = {
    ("corn", "beginning stocks"): {
        "label": "Beginning Stocks",
        "section": "Supply",
    },
    ("corn", "feed residual"): {
        "label": "Feed & Residual",
        "section": "Domestic Use",
    },
}


def normalize(value):
    value = str(value).strip().lower()
    value = re.sub(r"[_-]+", " ", value)
    return re.sub(r"\s+", " ", value)


def map_wasde_item(commodity, raw_item):
    commodity = normalize(commodity)
    raw_item = normalize(raw_item)

    mapping = WASDE_MAP.get((commodity, raw_item))

    if mapping:
        return {
            "label": mapping["label"],
            "section": mapping["section"],
            "status": "Verified",
        }

    # Readable fallback, but do not invent a section.
    return {
        "label": raw_item.title(),
        "section": "Other Items",
        "status": "Review",
    }


In [322]:
print(
    audit.groupby("commodity")["snd_item"]
    .nunique()
    .sort_values(ascending=False)
)

commodity
cotton          18
wheat           18
corn            17
soybeans        17
barley          15
oats            15
feed grains     14
soybean oil     14
soybean meal    12
coarse grain     8
Name: snd_item, dtype: int64


In [324]:

commodities = ["corn", "soybeans", "wheat"]

audit = (
    df[df["commodity"].isin(commodities)]
    [["commodity", "country", "table", "snd_item"]]
    .drop_duplicates()
    .sort_values(["commodity", "country", "table", "snd_item"])
)

print(audit.to_string(index=False))

audit.to_csv("wasde_core_commodities_audit.csv", index=False)


commodity          country table               snd_item
     corn        argentina    ww       beginning stocks
     corn        argentina    ww domestic feed or crush
     corn        argentina    ww         domestic total
     corn        argentina    ww          ending stocks
     corn        argentina    ww                exports
     corn        argentina    ww                imports
     corn        argentina    ww             production
     corn        argentina    ww           stock to use
     corn           brazil    ww       beginning stocks
     corn           brazil    ww domestic feed or crush
     corn           brazil    ww         domestic total
     corn           brazil    ww          ending stocks
     corn           brazil    ww                exports
     corn           brazil    ww                imports
     corn           brazil    ww             production
     corn           brazil    ww           stock to use
     corn           canada    ww       beginning

In [326]:




# Display labels
ITEM_LABELS = {
    "beginning stocks": "Beginning Stocks",
    "production": "Production",
    "imports": "Imports",
    "supply total": "Total Supply",
    "domestic total": "Domestic Use",
    "domestic use": "Domestic Use",
    "domestic feed or crush": "Domestic Feed / Crush",
    "crushing": "Crushing",
    "ethanol": "Ethanol",
    "feed residual": "Feed & Residual",
    "food seed industrial": "Food, Seed & Industrial",
    "food feed industrial": "Food, Feed & Industrial",
    "food": "Food",
    "seed": "Seed",
    "residual": "Residual",
    "exports": "Exports",
    "use total": "Total Use",
    "ending stocks": "Ending Stocks",
    "stock to use": "Stocks-to-Use",
    "planted area": "Area Planted",
    "harvested area": "Area Harvested",
    "yield": "Yield",
    "avg. farm price": "Avg. Farm Price",
}

df["item_label"] = (
    df["snd_item"]
    .map(ITEM_LABELS)
    .fillna(df["snd_item"].str.title())
)

In [327]:
SECTION_MAP = {
    "beginning stocks": "Beginning Stocks",
    "production": "Supply",
    "imports": "Supply",
    "supply total": "Supply",
    "planted area": "Area",
    "harvested area": "Area",
    "yield": "Area",
    "domestic total": "Use",
    "domestic use": "Use",
    "domestic feed or crush": "Use",
    "crushing": "Use",
    "ethanol": "Use",
    "feed residual": "Use",
    "food seed industrial": "Use",
    "food feed industrial": "Use",
    "food": "Use",
    "seed": "Use",
    "residual": "Use",
    "exports": "Use",
    "use total": "Use",
    "ending stocks": "Ending Stocks",
    "stock to use": "Key Ratios",
    "avg. farm price": "Price",
}

df["section"] = df["snd_item"].map(SECTION_MAP).fillna("Other")

In [328]:
unmapped = (
    df.loc[~df["snd_item"].isin(ITEM_LABELS), "snd_item"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

print(unmapped.to_list())

['avg. price', 'bio diesel', 'domestic disappearance', 'exports total', 'harvested', 'loss', 'planted', 'total domestic', 'unaccounted']


In [ ]:

import pandas as pd

df = pd.read_csv(FILE_PATH)

ITEM_LABELS = {
    # Stocks
    "beginning stocks": "Beginning Stocks",
    "ending stocks": "Ending Stocks",
    "stock to use": "Stocks-to-Use",

    # Supply
    "production": "Production",
    "imports": "Imports",
    "supply total": "Total Supply",

    # Domestic use
    "domestic total": "Domestic Use",
    "total domestic": "Total Domestic Use",
    "domestic use": "Domestic Use",
    "domestic disappearance": "Domestic Disappearance",
    "domestic feed or crush": "Domestic Feed / Crush",
    "feed residual": "Feed & Residual",
    "food seed industrial": "Food, Seed & Industrial",
    "food feed industrial": "Food, Feed & Industrial",
    "food": "Food",
    "seed": "Seed",
    "residual": "Residual",
    "crushing": "Crushing",
    "ethanol": "Ethanol",
    "bio diesel": "Biodiesel",
    "use total": "Total Use",

    # Trade and adjustments
    "exports": "Exports",
    "exports total": "Total Exports",
    "loss": "Loss",
    "unaccounted": "Unaccounted",

    # Area and yield
    "planted area": "Area Planted",
    "harvested area": "Area Harvested",
    "planted": "Planted",
    "harvested": "Harvested",
    "yield": "Yield",

    # Prices
    "avg. farm price": "Avg. Farm Price",
    "avg. price": "Avg. Price",
}

#clean snd_items
df["snd_item_clean"] = (
    df["snd_item"]
    .astype("string")
    .str.strip()
    .str.lower()
)


# APPLY DISPLAY LABELS

df["item_label"] = (
    df["snd_item_clean"]
    .map(ITEM_LABELS)
    .fillna(df["snd_item"].astype("string").str.strip().str.title())
)


# ============================================================
# 5. SECTION MAPPING
# ============================================================

SECTION_MAP = {
    # Stocks
    "beginning stocks": "Beginning Stocks",
    "ending stocks": "Ending Stocks",
    "stock to use": "Key Ratios",

    # Supply
    "production": "Supply",
    "imports": "Supply",
    "supply total": "Supply",

    # Area and yield
    "planted area": "Area",
    "harvested area": "Area",
    "planted": "Area",
    "harvested": "Area",
    "yield": "Area",

    # Domestic use
    "domestic total": "Use",
    "total domestic": "Use",
    "domestic use": "Use",
    "domestic disappearance": "Use",
    "domestic feed or crush": "Use",
    "feed residual": "Use",
    "food seed industrial": "Use",
    "food feed industrial": "Use",
    "food": "Use",
    "seed": "Use",
    "residual": "Use",
    "crushing": "Use",
    "ethanol": "Use",
    "bio diesel": "Use",
    "use total": "Use",

    # Trade and adjustments
    "exports": "Use",
    "exports total": "Use",
    "loss": "Other",
    "unaccounted": "Other",

    # Prices
    "avg. farm price": "Price",
    "avg. price": "Price",
}

df["section"] = (
    df["snd_item_clean"]
    .map(SECTION_MAP)
    .fillna("Other")
)


# ============================================================
# 6. DEFINE SECTION AND ITEM DISPLAY ORDER
# ============================================================

SECTION_ORDER = [
    "Area",
    "Beginning Stocks",
    "Supply",
    "Use",
    "Ending Stocks",
    "Key Ratios",
    "Price",
    "Other",
]

ITEM_ORDER = [
    "Area Planted",
    "Planted",
    "Area Harvested",
    "Harvested",
    "Yield",
    "Beginning Stocks",
    "Production",
    "Imports",
    "Total Supply",
    "Domestic Feed / Crush",
    "Crushing",
    "Feed & Residual",
    "Food",
    "Seed",
    "Residual",
    "Ethanol",
    "Biodiesel",
    "Food, Seed & Industrial",
    "Food, Feed & Industrial",
    "Domestic Use",
    "Domestic Disappearance",
    "Total Domestic Use",
    "Exports",
    "Total Exports",
    "Total Use",
    "Loss",
    "Unaccounted",
    "Ending Stocks",
    "Stocks-to-Use",
    "Avg. Farm Price",
    "Avg. Price",
]


# ============================================================
# 7. SORT WITHOUT LOSING UNLISTED ITEMS
# ============================================================

section_order_map = {
    name: i for i, name in enumerate(SECTION_ORDER)
}

item_order_map = {
    name: i for i, name in enumerate(ITEM_ORDER)
}

df["_section_order"] = (
    df["section"]
    .map(section_order_map)
    .fillna(len(SECTION_ORDER))
)

df["_item_order"] = (
    df["item_label"]
    .map(item_order_map)
    .fillna(len(ITEM_ORDER))
)

df = (
    df.sort_values(
        ["_section_order", "_item_order", "item_label"],
        kind="stable",
    )
    .drop(columns=["_section_order", "_item_order"])
)


# ============================================================
# 8. CHECK FOR UNMAPPED ITEMS
# ============================================================

unmapped = sorted(
    set(df["snd_item_clean"].dropna().unique())
    - set(ITEM_LABELS.keys())
)

print("Unmapped WASDE items:")
print(unmapped)


# ============================================================
# 9. AUDIT UNIQUE ITEMS BY COMMODITY / COUNTRY / TABLE
# ============================================================

item_audit = (
    df[
        [
            "commodity",
            "country",
            "table",
            "snd_item",
            "item_label",
            "section",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["commodity", "country", "table", "section", "item_label"],
        kind="stable",
    )
)

print("\nWASDE item audit:")
print(item_audit.to_string(index=False))


# ============================================================
# 10. EXAMPLE: US CORN BALANCE SHEET ITEMS
# ============================================================

corn_us = df[
    (df["commodity"] == "corn")
    & (df["country"] == "united states")
    & (df["table"] == "us")
].copy()

corn_us_items = (
    corn_us[["section", "item_label", "snd_item"]]
    .drop_duplicates()
    .sort_values(
        ["section", "item_label"],
        kind="stable",
    )
)

print("\nUS Corn balance sheet items:")
print(corn_us_items.to_string(index=False))


Unmapped WASDE items:
[]

WASDE item audit:
   commodity          country table               snd_item              item_label          section
      barley    united states    us         harvested area          Area Harvested             Area
      barley    united states    us           planted area            Area Planted             Area
      barley    united states    us                  yield                   Yield             Area
      barley    united states    us       beginning stocks        Beginning Stocks Beginning Stocks
      barley    united states    us          ending stocks           Ending Stocks    Ending Stocks
      barley    united states    us           stock to use           Stocks-to-Use       Key Ratios
      barley    united states    us        avg. farm price         Avg. Farm Price            Price
      barley    united states    us                imports                 Imports           Supply
      barley    united states    us             producti

In [333]:
df["snd_item"].unique()

<ArrowStringArray>
[          'planted area',                'planted',         'harvested area',
              'harvested',                  'yield',       'beginning stocks',
             'production',                'imports',           'supply total',
 'domestic feed or crush',               'crushing',          'feed residual',
                   'food',                   'seed',               'residual',
                'ethanol',             'bio diesel',   'food seed industrial',
   'food feed industrial',           'domestic use',         'domestic total',
 'domestic disappearance',         'total domestic',                'exports',
          'exports total',              'use total',          'ending stocks',
           'stock to use',        'avg. farm price',             'avg. price',
                   'loss',            'unaccounted']
Length: 32, dtype: str

In [335]:
import pandas as pd


# Normalize column names, not the source data values
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Summarize where each original item appears
group_cols = [
    col for col in [
        "snd_item",
        "commodity",
        "country",
        "table",
        "wheat_class",
    ]
    if col in df.columns
]

audit = (
    df.groupby(group_cols, dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values(["snd_item", "commodity", "country"])
)

display(audit)

,snd_item,commodity,country,table,wheat_class,row_count
0,avg. farm price,barley,united states,us,NaN,44
1,avg. farm price,corn,united states,us,NaN,46
2,avg. farm price,cotton,united states,us,NaN,46
3,avg. farm price,oats,united states,us,NaN,44
4,avg. farm price,soybean meal,united states,us,NaN,2
...,...,...,...,...,...,...
1128,yield,cotton,united states,us,NaN,388
1129,yield,feed grains,united states,us,NaN,44
1130,yield,oats,united states,us,NaN,44
1131,yield,soybeans,united states,us,NaN,388


In [340]:
import pandas as pd

# Create a DataFrame and set 1-based indexing
df_items = pd.DataFrame({"snd_item": items}, index=range(1, len(items) + 1))

# Ensure pandas doesn't truncate rows with '...'
pd.set_option('display.max_rows', None)

display(df_items)

,snd_item
1,avg. farm price
2,avg. price
3,beginning stocks
4,bio diesel
5,crushing
6,domestic disappearance
7,domestic feed or crush
8,domestic total
9,domestic use
10,ending stocks


In [10]:

import pandas as pd
from pathlib import Path


# ============================================================
# 1. DISPLAY ORDER — 32 SOURCE ITEMS
# ============================================================

MONTH_ABBR = (
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
)

AREA_YIELD_ITEMS = [
    "planted area",
    "harvested area",
    "harvested",
    "planted",
    "yield",
]

SUPPLY_ITEMS = [
    "beginning stocks",
    "production",
    "imports",
    "supply total",
]

COMMON_USE_ITEMS = [
    "avg. farm price",
    "avg. price",
    "bio diesel",
    "crushing",
    "domestic disappearance",
    "domestic feed or crush",
    "domestic total",
    "domestic use",
    "ending stocks",
    "ethanol",
    "exports",
    "exports total",
    "feed residual",
    "food",
    "food feed industrial",
    "food seed industrial",
    "loss",
    "residual",
    "seed",
    "stock to use",
    "total domestic",
    "unaccounted",
    "use total",
]

WASDE_ITEM_ORDER = {
    "default": AREA_YIELD_ITEMS + SUPPLY_ITEMS + COMMON_USE_ITEMS,
}


# ============================================================
# 2. DISPLAY LABELS
# ============================================================

WASDE_ITEM_LABELS = {
    "planted area": "Planted Area",
    "planted": "Planted",
    "harvested area": "Harvested Area",
    "harvested": "Harvested",
    "yield": "Yield",
    "beginning stocks": "Beginning Stocks",
    "production": "Production",
    "imports": "Imports",
    "supply total": "Supply Total",
    "avg. farm price": "Avg. Farm Price",
    "avg. price": "Avg. Price",
    "bio diesel": "Bio Diesel",
    "crushing": "Crushing",
    "domestic disappearance": "Domestic Disappearance",
    "domestic feed or crush": "Domestic Feed or Crush",
    "domestic total": "Domestic Total",
    "domestic use": "Domestic Use",
    "ending stocks": "Ending Stocks",
    "ethanol": "Ethanol",
    "exports": "Exports",
    "exports total": "Exports Total",
    "feed residual": "Feed Residual",
    "food": "Food",
    "food feed industrial": "Food Feed Industrial",
    "food seed industrial": "Food Seed Industrial",
    "loss": "Loss",
    "residual": "Residual",
    "seed": "Seed",
    "stock to use": "Stock to Use",
    "total domestic": "Total Domestic",
    "unaccounted": "Unaccounted",
    "use total": "Use Total",
}


def format_wasde_item(item_name):
    """Format an item for display without changing its source value."""
    normalized = str(item_name).strip().lower()

    return WASDE_ITEM_LABELS.get(
        normalized,
        str(item_name).replace("_", " ").strip().title(),
    )


# ============================================================
# 3. ITEM ORDERING AND MARKETING-YEAR HELPERS
# ============================================================

def order_wasde_items(item_names, commodity_name=None):
    """Put preferred items first and retain any additional items."""
    items = list(item_names)

    preferred_order = WASDE_ITEM_ORDER.get(
        str(commodity_name).strip().lower()
        if commodity_name is not None
        else "default",
        WASDE_ITEM_ORDER["default"],
    )

    ranks = {
        item.strip().lower(): position
        for position, item in enumerate(preferred_order)
    }

    return sorted(
        items,
        key=lambda item: (
            0, ranks[str(item).strip().lower()]
        )
        if str(item).strip().lower() in ranks
        else (1, str(item).strip().lower()),
    )


def parse_month_offset(month_offset):
    """Convert '1_10' into (1, 10)."""
    year_offset, calendar_month = str(month_offset).split("_")
    year_offset = int(year_offset)
    calendar_month = int(calendar_month)

    if year_offset < 0:
        raise ValueError(f"Invalid year offset: {year_offset}")

    if not 1 <= calendar_month <= 12:
        raise ValueError(f"Invalid calendar month: {calendar_month}")

    return year_offset, calendar_month


def format_marketing_year(crop_year, show_full_year=False):
    """Convert 2025 to '2025/26' or '2025/2026'."""
    crop_year = int(crop_year)
    next_year = crop_year + 1

    if show_full_year:
        return f"{crop_year}/{next_year}"

    return f"{crop_year}/{next_year % 100:02d}"


def format_balance_sheet_index(balance_sheet):
    """Apply display labels to the row index."""
    result = balance_sheet.copy()
    result.index = [format_wasde_item(item) for item in result.index]
    result.index.name = "WASDE Item"
    return result


def order_balance_sheet_rows(balance_sheet, commodity_name=None):
    """Apply the preferred item order."""
    if balance_sheet.empty:
        return balance_sheet

    ordered_names = order_wasde_items(
        balance_sheet.index,
        commodity_name,
    )
    return balance_sheet.reindex(ordered_names)


# ============================================================
# 4. PREPARE WASDE DATA
# ============================================================

def prepare_wasde_data(dataframe):
    """Normalize column names, text values, dates, and numeric values."""
    df = dataframe.copy()

    df.columns = [
        str(column).strip().lower().replace(" ", "_")
        for column in df.columns
    ]

    # Support common source-column naming variations.
    aliases = {
        "reportdate": "report_date",
        "marketyear": "crop_year",
        "cropyear": "crop_year",
        "monthoffset": "month_offset",
        "snditem": "snd_item",
    }
    df = df.rename(columns=aliases)

    required = [
        "report_date",
        "value",
        "crop_year",
        "commodity",
        "country",
        "snd_item",
        "table",
    ]

    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(
            f"Missing required columns: {missing}\n"
            f"Available columns: {df.columns.tolist()}"
        )

    df["report_date"] = pd.to_datetime(
        df["report_date"], errors="coerce"
    )
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df["crop_year"] = pd.to_numeric(df["crop_year"], errors="coerce")

    text_columns = [
        "commodity",
        "country",
        "snd_item",
        "table",
        "crop",
        "wheat_class",
    ]

    for col in text_columns:
        if col in df.columns:
            df[col] = (
                df[col].astype("string").str.strip().str.lower()
            )

    # Parse month offsets such as 0_05 or 1_10.
    if "month_offset" in df.columns:
        parts = df["month_offset"].astype("string").str.extract(
            r"^\s*(\d+)\s*_\s*(\d{1,2})\s*$"
        )

        df["year_off"] = pd.to_numeric(parts[0], errors="coerce")
        df["month_num"] = pd.to_numeric(parts[1], errors="coerce")
    else:
        df["year_off"] = pd.NA
        df["month_num"] = pd.NA
        df["month_offset"] = pd.NA

    # Keep yearly observations even when month_offset is missing.
    df = df.dropna(
        subset=[
            "report_date",
            "value",
            "crop_year",
            "commodity",
            "country",
            "snd_item",
            "table",
        ]
    ).copy()

    df["crop_year"] = df["crop_year"].astype(int)

    # Normalize month offsets to e.g. 0_05, 1_10.
    valid_offsets = (
        df["year_off"].notna()
        & df["month_num"].between(1, 12)
    )

    df["month_offset"] = pd.Series(
        pd.NA, index=df.index, dtype="string"
    )

    df.loc[valid_offsets, "month_offset"] = (
        df.loc[valid_offsets, "year_off"].astype(int).astype(str)
        + "_"
        + df.loc[valid_offsets, "month_num"].astype(int).astype(str).str.zfill(2)
    )

    df["year_off"] = pd.to_numeric(
        df["year_off"], errors="coerce"
    ).astype("Int64")

    df["month_num"] = pd.to_numeric(
        df["month_num"], errors="coerce"
    ).astype("Int64")

    return df


# ============================================================
# 5. FILTER WASDE DATA
# ============================================================

def select_wasde_data(
    dataframe,
    commodity,
    country,
    table=None,
    crop=None,
    wheat_class=None,
):
    """Filter by commodity and country, with optional series filters."""
    df = dataframe.copy()

    mask = (
        df["commodity"].eq(str(commodity).strip().lower()).fillna(False)
        & df["country"].eq(str(country).strip().lower()).fillna(False)
    )

    if table is not None:
        mask &= df["table"].eq(str(table).strip().lower()).fillna(False)

    if crop is not None and "crop" in df.columns:
        mask &= df["crop"].eq(str(crop).strip().lower()).fillna(False)

    if wheat_class is not None and "wheat_class" in df.columns:
        mask &= (
            df["wheat_class"]
            .eq(str(wheat_class).strip().lower())
            .fillna(False)
        )

    return df.loc[mask].copy()


# ============================================================
# 6. SHARED HELPERS
# ============================================================

def get_latest_records(dataframe, key_columns):
    """Keep the latest publication for each unique observation."""
    if dataframe.empty:
        return dataframe.copy()

    required = ["report_date", *key_columns]
    missing = [col for col in required if col not in dataframe.columns]

    if missing:
        raise ValueError(
            f"Missing columns for latest-record selection: {missing}"
        )

    return (
        dataframe
        .sort_values("report_date", kind="stable")
        .drop_duplicates(subset=key_columns, keep="last")
        .copy()
    )


def get_commodity_name(dataframe):
    if dataframe.empty or "commodity" not in dataframe.columns:
        return None

    names = dataframe["commodity"].dropna()
    return None if names.empty else names.iloc[0]


def get_series_key_columns(dataframe):
    """Identify dimensions that contain multiple series."""
    candidates = ["crop", "wheat_class", "table"]

    return [
        col for col in candidates
        if col in dataframe.columns
        and dataframe[col].nunique(dropna=True) > 1
    ]


# ============================================================
# 7. YEARLY TABLE
# ============================================================

def build_yearly_table(
    dataframe,
    as_of="latest",
    first_year=None,
    last_year=None,
):
    """
    Rows: WASDE items.
    Columns: marketing years, e.g. 2024/25, 2025/26.

    as_of='latest': latest publication per item and crop year.
    as_of='2026-09-30': latest publication on or before that date.
    """
    df = dataframe.copy()

    if df.empty:
        return pd.DataFrame()

    if as_of != "latest":
        cutoff = pd.to_datetime(as_of, errors="raise")
        df = df[df["report_date"] <= cutoff].copy()

    if first_year is not None:
        df = df[df["crop_year"] >= int(first_year)]

    if last_year is not None:
        df = df[df["crop_year"] <= int(last_year)]

    if df.empty:
        return pd.DataFrame()

    commodity = get_commodity_name(df)
    series_keys = get_series_key_columns(df)

    keys = ["snd_item", "crop_year", *series_keys]
    latest = get_latest_records(df, keys)

    if latest.duplicated(["snd_item", "crop_year"]).any():
        raise ValueError(
            "Multiple WASDE series remain for the same item and crop year. "
            "Filter by table, crop, or wheat_class."
        )

    # No aggfunc: unexpected duplicate rows must not be aggregated.
    result = latest.pivot(
        index="snd_item",
        columns="crop_year",
        values="value",
    )

    result = result.reindex(sorted(result.columns), axis=1)

    result.columns = [
        format_marketing_year(year)
        for year in result.columns
    ]
    result.columns.name = "Marketing Year"

    result = order_balance_sheet_rows(result, commodity)
    return format_balance_sheet_index(result)


# ============================================================
# 8. MONTHLY TABLE
# ============================================================

def build_monthly_table(dataframe, crop_year):
    """
    Rows: WASDE items.
    Columns: monthly observations for the selected marketing year.

    For each item/month, retain the latest publication available.
    """
    df = dataframe.copy()

    if df.empty:
        return pd.DataFrame()

    df = df[
        df["crop_year"].eq(int(crop_year))
        & df["month_offset"].notna()
    ].copy()

    if df.empty:
        return pd.DataFrame()

    # Recreate offset fields to ensure they're numeric.
    parts = df["month_offset"].astype("string").str.extract(
        r"^\s*(\d+)\s*_\s*(\d{1,2})\s*$"
    )

    df["year_off"] = pd.to_numeric(parts[0], errors="coerce")
    df["month_num"] = pd.to_numeric(parts[1], errors="coerce")

    df = df.dropna(subset=["year_off", "month_num"]).copy()
    df = df[df["month_num"].between(1, 12)].copy()

    if df.empty:
        return pd.DataFrame()

    df["year_off"] = df["year_off"].astype(int)
    df["month_num"] = df["month_num"].astype(int)

    commodity = get_commodity_name(df)
    series_keys = get_series_key_columns(df)

    keys = ["snd_item", "month_offset", *series_keys]
    latest = get_latest_records(df, keys)

    if latest.duplicated(["snd_item", "month_offset"]).any():
        raise ValueError(
            "Multiple WASDE series remain for the same item and month. "
            "Filter by table, crop, or wheat_class."
        )

    month_order = (
        latest[["month_offset", "year_off", "month_num"]]
        .drop_duplicates()
        .sort_values(["year_off", "month_num"])
    )

    offsets = month_order["month_offset"].tolist()

    # No aggfunc: pivot raises if duplicate observations remain.
    result = latest.pivot(
        index="snd_item",
        columns="month_offset",
        values="value",
    )

    result = result.reindex(columns=offsets)

    month_labels = []
    for _, row in month_order.iterrows():
        calendar_year = int(crop_year) + int(row["year_off"])
        month = MONTH_ABBR[int(row["month_num"]) - 1]
        month_labels.append(f"{month}-{calendar_year % 100:02d}")

    result.columns = month_labels
    result.columns.name = (
        f"Marketing Year {format_marketing_year(crop_year)}"
    )

    result = order_balance_sheet_rows(result, commodity)
    return format_balance_sheet_index(result)


# ============================================================
# 9. LOAD DATA AND DISPLAY TABLES
# ============================================================

FILE_PATH = Path("../data/wasde/wasde_processed_all_historicals.csv")

raw_df = pd.read_csv(FILE_PATH)
df = prepare_wasde_data(raw_df)

print("Available tables:")
print(sorted(df["table"].dropna().unique()))

print("\nAvailable commodities:")
print(sorted(df["commodity"].dropna().unique()))

print("\nAvailable countries:")
print(sorted(df["country"].dropna().unique()))

# Filter US corn.
# Change table="us" if your CSV uses a different table label.
corn_us = select_wasde_data(
    df,
    commodity="corn",
    country="united states",
    table="us",
)

print("\nUS corn record count:", len(corn_us))

if corn_us.empty:
    print("No rows found. Check the available country and table values.")
else:
    # Latest yearly values
    print("\nYEARLY BALANCE SHEET — LATEST")
    yearly_table = build_yearly_table(corn_us)
    display(yearly_table)

    # Latest yearly values available as of 30 September 2026
    print("\nYEARLY BALANCE SHEET — AS OF 2026-09-30")
    yearly_as_of = build_yearly_table(
        corn_us,
        as_of="2026-09-30",
    )
    display(yearly_as_of)

    # Monthly observations for marketing year 2025/26
    print("\nMONTHLY BALANCE SHEET — 2025/26")
    monthly_table = build_monthly_table(
        corn_us,
        crop_year=2025,
    )
    display(monthly_table)


Available tables:
['bc', 'us', 'ww']

Available commodities:
['barley', 'coarse grain', 'corn', 'cotton', 'feed grains', 'oats', 'soybean meal', 'soybean oil', 'soybeans', 'wheat']

Available countries:
['afr. fr. zone', 'argentina', 'australia', 'bangladesh', 'brazil', 'canada', 'central asia', 'china', 'egypt', 'eu', 'india', 'indonesia', 'japan', 'kazakhstan', 'major exporters', 'major importers', 'mexico', 'n. afr & mideast', 'nigeria', 'north africa', 'pakistan', 'paraguay', 'russia', 's. hemis.', 'saudi arabia', 'sel mideast', 'south africa', 'south korea', 'southeast asia', 'thailand', 'total foreign', 'turkey', 'ukraine', 'united kingdom', 'united states', 'vietnam', 'world', 'world less china']

US corn record count: 5844

YEARLY BALANCE SHEET — LATEST


ValueError: Multiple WASDE series remain for the same item and crop year. Filter by table, crop, or wheat_class.

In [11]:

# Check which dimensions contain multiple series
for col in ["table", "crop", "wheat_class"]:
    if col in corn_us.columns:
        print(f"\n{col}:")
        print(corn_us[col].value_counts(dropna=False).head(20))

# Find items and crop years with multiple rows
duplicates = corn_us[
    corn_us.duplicated(
        subset=["snd_item", "crop_year"],
        keep=False,
    )
].sort_values(["snd_item", "crop_year", "report_date"])

display(
    duplicates[
        [
            "report_date",
            "commodity",
            "country",
            "table",
            "crop",
            "wheat_class",
            "snd_item",
            "crop_year",
            "value",
        ]
    ].head(50)
)



table:
table
us    5844
Name: count, dtype: int64[pyarrow]

crop:
crop
oc    2922
nc    2922
Name: count, dtype: int64[pyarrow]

wheat_class:
wheat_class
<NA>    5844
Name: count, dtype: int64[pyarrow]


,report_date,commodity,country,table,crop,wheat_class,snd_item,crop_year,value
152416,2024-10-11,corn,united states,us,oc,<NA>,avg. farm price,2023,4.55
154647,2024-11-24,corn,united states,us,oc,<NA>,avg. farm price,2023,4.55
156878,2024-12-10,corn,united states,us,oc,<NA>,avg. farm price,2023,4.55
159109,2025-01-10,corn,united states,us,oc,<NA>,avg. farm price,2023,4.55
161340,2025-02-11,corn,united states,us,oc,<NA>,avg. farm price,2023,4.55
163571,2025-03-11,corn,united states,us,oc,<NA>,avg. farm price,2023,4.55
165802,2025-04-10,corn,united states,us,oc,<NA>,avg. farm price,2023,4.55
152417,2024-10-11,corn,united states,us,nc,<NA>,avg. farm price,2024,4.10
154648,2024-11-24,corn,united states,us,nc,<NA>,avg. farm price,2024,4.10
156879,2024-12-10,corn,united states,us,nc,<NA>,avg. farm price,2024,4.10


In [12]:
from pathlib import Path

file_path = Path("../data/wasde/wasde_processed_all_historicals.csv")

df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nAvailable commodities:")
print(sorted(df["commodity"].dropna().unique()))

print("\nAvailable countries:")
print(sorted(df["country"].dropna().unique()))

print("\nAvailable tables:")
print(sorted(df["table"].dropna().unique()))

display(df.head())


Shape: (202960, 13)

Columns:
['Report Date', 'Day', 'Month', 'Year', 'Value', 'commodity', 'country', 'snd_item', 'wheat_class', 'crop', 'table', 'crop year', 'month offset']

Available commodities:
['barley', 'coarse grain', 'corn', 'cotton', 'feed grains', 'oats', 'soybean meal', 'soybean oil', 'soybeans', 'wheat']

Available countries:
['afr. fr. zone', 'argentina', 'australia', 'bangladesh', 'brazil', 'canada', 'central asia', 'china', 'egypt', 'eu', 'india', 'indonesia', 'japan', 'kazakhstan', 'major exporters', 'major importers', 'mexico', 'n. afr & mideast', 'nigeria', 'north africa', 'pakistan', 'paraguay', 'russia', 's. hemis.', 'saudi arabia', 'sel mideast', 'south africa', 'south korea', 'southeast asia', 'thailand', 'total foreign', 'turkey', 'ukraine', 'united kingdom', 'united states', 'vietnam', 'world', 'world less china']

Available tables:
['bc', 'us', 'ww']


,Report Date,Day,Month,Year,Value,commodity,country,snd_item,wheat_class,crop,table,crop year,month offset
0,2024/1/12,12,1,2024,24.0,wheat,united states,beginning stocks,durum,oc,bc,2022,2_01
1,2024/1/12,12,1,2024,88.0,wheat,united states,domestic use,durum,oc,bc,2022,2_01
2,2024/1/12,12,1,2024,28.0,wheat,united states,ending stocks,durum,oc,bc,2022,2_01
3,2024/1/12,12,1,2024,23.0,wheat,united states,exports,durum,oc,bc,2022,2_01
4,2024/1/12,12,1,2024,0.0,wheat,united states,feed residual,durum,oc,bc,2022,2_01


In [28]:

from pathlib import Path
import pandas as pd
import re

FILE_PATH = Path("../data/wasde/wasde_processed_all_historicals.csv")

df = pd.read_csv(FILE_PATH)

# Normalize column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Convert data types
df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce")
df["crop_year"] = pd.to_numeric(df["crop_year"], errors="coerce")
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Normalize descriptive columns
for col in ["commodity", "country", "snd_item", "wheat_class", "crop", "table"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip().str.lower()

# Preserve month_offset as a string identifier
df["month_offset"] = df["month_offset"].astype("string").str.strip()

# Extract year offset and month number
parts = df["month_offset"].str.extract(
    r"^\s*(\d+)\s*_\s*(\d{1,2})\s*$"
)

df["year_offset"] = pd.to_numeric(parts[0], errors="coerce")
df["offset_month"] = pd.to_numeric(parts[1], errors="coerce")
df["crop_year"] = df["crop_year"].astype(int)
df["year_offset"] = df["year_offset"].astype(int)
df["offset_month"] = df["offset_month"].astype(int)

df = df[df["offset_month"].between(1, 12)].copy()

print("Rows:", len(df))
print("Available commodities:", sorted(df["commodity"].dropna().unique()))


Rows: 202960
Available commodities: ['barley', 'coarse grain', 'corn', 'cotton', 'feed grains', 'oats', 'soybean meal', 'soybean oil', 'soybeans', 'wheat']


In [21]:

def format_marketing_year(year):
    """2025 -> 2025/26."""
    year = int(year)
    return f"{year}/{(year + 1) % 100:02d}"


def format_month_offset(month_offset):
    """0_05 -> May (Offset 0); 1_05 -> May (Offset 1)."""
    year_offset, month = map(int, str(month_offset).split("_"))
    month_name = MONTH_ABBR[month - 1]
    return f"{month_name} (Offset {year_offset})"


def format_wasde_item(item):
    """Apply the existing WASDE item labels."""
    item = str(item).strip().lower()
    return WASDE_ITEM_LABELS.get(
        item,
        item.replace("_", " ").title()
    )


def order_wasde_rows(table):
    """Apply the existing WASDE display order."""
    item_order = {
        format_wasde_item(item): position
        for position, item in enumerate(WASDE_ITEM_ORDER["default"])
    }

    return table.reindex(
        sorted(
            table.index,
            key=lambda item: (
                item_order.get(str(item), len(item_order)),
                str(item),
            ),
        )
    )


In [22]:

COMMODITY = "corn"
COUNTRY = "united states"
TABLE = "us"

data = df.loc[
    (df["commodity"] == COMMODITY)
    & (df["country"] == COUNTRY)
    & (df["table"] == TABLE)
].copy()

if data.empty:
    raise ValueError("No data found for the selected commodity, country and table.")

# Do not add crop or wheat_class to the row labels.
# Check whether the selected dimensions contain duplicate table cells.
print("Records:", len(data))
print("Marketing years:", data["crop_year"].nunique())
print("WASDE items:", data["snd_item"].nunique())


Records: 5844
Marketing years: 18
WASDE items: 16


In [29]:

# Available month-offset choices
available_offsets = sorted(
    data["month_offset"].dropna().unique(),
    key=lambda offset: tuple(map(int, str(offset).split("_")))
)

offset_labels = {
    offset: format_month_offset(offset)
    for offset in available_offsets
}

print("Available month offsets:")
for offset in available_offsets:
    print(f"{offset}: {offset_labels[offset]}")

# Select the desired position
SELECTED_MONTH_OFFSET = "0_09"

if SELECTED_MONTH_OFFSET not in available_offsets:
    raise ValueError(
        f"{SELECTED_MONTH_OFFSET} is not available. "
        f"Choose from: {available_offsets}"
    )

yearly_data = data.loc[
    data["month_offset"] == SELECTED_MONTH_OFFSET
].copy()

# Report Date identifies the publication.
# Keep the most recent publication only if the exact same
# item/marketing-year/offset has multiple publication records.
keys = ["snd_item", "crop_year", "month_offset"]

yearly_data = (
    yearly_data
    .sort_values("report_date")
    .drop_duplicates(subset=keys, keep="last")
)

# Check for duplicate cells after removing OC/NC distinctions.
duplicates = yearly_data.duplicated(
    subset=["snd_item", "crop_year"],
    keep=False,
)

if duplicates.any():
    display(
        yearly_data.loc[
            duplicates,
            ["report_date", "snd_item", "crop_year", "crop", "value", "month_offset"]
        ].sort_values(["snd_item", "crop_year", "report_date"])
    )
    raise ValueError(
        "Duplicate cells remain for the yearly table. "
        "Multiple values exist for an item and marketing year at this offset. "
        "Inspect the displayed records; no value has been silently selected."
    )

yearly_table = yearly_data.pivot(
    index="snd_item",
    columns="crop_year",
    values="value",
)

yearly_table.columns = [
    format_marketing_year(year)
    for year in yearly_table.columns
]

yearly_table.columns.name = "Marketing Year"
yearly_table.index = [
    format_wasde_item(item)
    for item in yearly_table.index
]
yearly_table.index.name = "WASDE Item"

yearly_table = order_wasde_rows(yearly_table)

display(yearly_table.round(2))


Available month offsets:
0_05: May (Offset 0)
0_06: Jun (Offset 0)
0_07: Jul (Offset 0)
0_08: Aug (Offset 0)
0_09: Sep (Offset 0)
0_10: Oct (Offset 0)
0_11: Nov (Offset 0)
0_12: Dec (Offset 0)
1_01: Jan (Offset 1)
1_02: Feb (Offset 1)
1_03: Mar (Offset 1)
1_04: Apr (Offset 1)
1_05: May (Offset 1)
1_06: Jun (Offset 1)
1_07: Jul (Offset 1)
1_08: Aug (Offset 1)
1_09: Sep (Offset 1)
1_10: Oct (Offset 1)
1_11: Nov (Offset 1)
1_12: Dec (Offset 1)
2_01: Jan (Offset 2)
2_02: Feb (Offset 2)
2_03: Mar (Offset 2)
2_04: Apr (Offset 2)


Marketing Year,2010/11,2011/12,2012/13,2013/14,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25,2025/26,2026/27
WASDE Item,,,,,,,,,,,,,,,,,
Planted Area,87.90,92.30,96.40,97.40,91.60,88.90,94.10,90.90,89.10,90.00,92.00,93.3,88.60,94.90,90.70,98.70,96.8
Harvested Area,81.00,84.40,87.40,89.10,83.80,81.10,86.60,83.50,81.80,82.00,83.50,85.1,80.80,87.10,82.70,90.00,88.5
Yield,162.50,148.10,122.80,155.30,171.70,167.50,174.40,169.90,181.30,168.20,178.50,176.3,172.50,173.80,183.60,186.70,178.5
Beginning Stocks,1386.00,920.00,1181.00,661.00,1181.00,1732.00,1716.00,2350.00,2002.00,2445.00,2253.00,1187.0,1525.00,1452.00,1812.00,1325.00,1922.0
Production,13160.00,12497.00,10727.00,13843.00,14395.00,13585.00,15093.00,14184.00,14827.00,13799.00,14900.00,14996.0,13944.00,15134.00,15186.00,16814.00,15800.0
Imports,10.00,15.00,75.00,25.00,30.00,30.00,50.00,50.00,50.00,50.00,25.00,25.0,25.00,25.00,25.00,25.00,25.0
Supply Total,14556.00,13432.00,11983.00,14530.00,15607.00,15347.00,16859.00,16585.00,16879.00,16295.00,17178.00,16208.0,15494.00,16611.00,17022.00,18165.00,17747.0
Avg. Farm Price,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.90,4.8
Domestic Total,11340.00,11110.00,10000.00,11450.00,11855.00,11905.00,12300.00,12400.00,12705.00,12055.00,12350.00,12325.0,12000.00,12340.00,12665.00,13080.00,12905.0


In [32]:

SELECTED_CROP_YEAR = 2025

monthly_data = data.loc[
    data["crop_year"] == SELECTED_CROP_YEAR
].copy()

if monthly_data.empty:
    raise ValueError(
        f"No data found for marketing year {format_marketing_year(SELECTED_CROP_YEAR)}."
    )

# One observation per item and month-offset position.
# If the same cell has multiple report dates, retain the latest publication.
keys = ["snd_item", "crop_year", "month_offset"]

monthly_data = (
    monthly_data
    .sort_values("report_date")
    .drop_duplicates(subset=keys, keep="last")
)

# Check for duplicate cells after removing OC/NC distinctions.
duplicates = monthly_data.duplicated(
    subset=["snd_item", "month_offset"],
    keep=False,
)

if duplicates.any():
    display(
        monthly_data.loc[
            duplicates,
            ["report_date", "snd_item", "crop_year", "crop", "value", "month_offset"]
        ].sort_values(["snd_item", "month_offset", "report_date"])
    )
    raise ValueError(
        "Duplicate cells remain for the monthly table. "
        "Inspect the displayed records; no value has been silently selected."
    )

monthly_table = monthly_data.pivot(
    index="snd_item",
    columns="month_offset",
    values="value",
)

# Order columns using the numerical offset year and month
ordered_offsets = sorted(
    monthly_table.columns,
    key=lambda offset: tuple(map(int, str(offset).split("_")))
)

monthly_table = monthly_table.reindex(columns=ordered_offsets)

monthly_table.columns = [
    format_month_offset(offset)
    for offset in monthly_table.columns
]

monthly_table.columns.name = (
    f"Marketing Year {format_marketing_year(SELECTED_CROP_YEAR)}"
)

monthly_table.index = [
    format_wasde_item(item)
    for item in monthly_table.index
]
monthly_table.index.name = "WASDE Item"

monthly_table = order_wasde_rows(monthly_table)

display(monthly_table.round(2))


TypeError: format_month_offset() missing 1 required positional argument: 'crop_year'

In [33]:

from pathlib import Path
import pandas as pd
import re


# ============================================================
# 1. LOAD AND CLEAN DATA
# ============================================================

FILE_PATH = Path("../data/wasde/wasde_processed_all_historicals.csv")

df = pd.read_csv(FILE_PATH)

df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

df["report_date"] = pd.to_datetime(
    df["report_date"], errors="coerce"
)

df["crop_year"] = pd.to_numeric(
    df["crop_year"], errors="coerce"
)

df["value"] = pd.to_numeric(
    df["value"], errors="coerce"
)

for col in [
    "commodity",
    "country",
    "snd_item",
    "wheat_class",
    "crop",
    "table",
]:
    if col in df.columns:
        df[col] = (
            df[col].astype("string").str.strip().str.lower()
        )

df["month_offset"] = (
    df["month_offset"].astype("string").str.strip()
)

# Extract offset year and month
parts = df["month_offset"].str.extract(
    r"^\s*(\d+)\s*_\s*(\d{1,2})\s*$"
)

df["year_offset"] = pd.to_numeric(
    parts[0], errors="coerce"
)

df["offset_month"] = pd.to_numeric(
    parts[1], errors="coerce"
)

df = df.dropna(
    subset=[
        "report_date",
        "crop_year",
        "value",
        "year_offset",
        "offset_month",
    ]
).copy()

df["crop_year"] = df["crop_year"].astype(int)
df["year_offset"] = df["year_offset"].astype(int)
df["offset_month"] = df["offset_month"].astype(int)

df = df[df["offset_month"].between(1, 12)].copy()

print("Rows:", len(df))
print(
    "Available commodities:",
    sorted(df["commodity"].dropna().unique()),
)


# ============================================================
# 2. DISPLAY FORMATTING
# ============================================================

MONTH_ABBR = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
]


def format_marketing_year(year):
    """2010 -> 2010/11."""
    year = int(year)
    return f"{year}/{(year + 1) % 100:02d}"


def format_month_offset(month_offset, crop_year):
    """
    Convert an offset to MMM YY.

    Example under the convention that crop_year is the
    marketing year's starting calendar year:
        0_05 -> May 10 for crop_year=2010
        0_10 -> Oct 10 for crop_year=2010
        1_01 -> Jan 11 for crop_year=2010
    """
    year_offset, month = map(
        int, str(month_offset).split("_")
    )

    calendar_year = int(crop_year) + year_offset

    return (
        f"{MONTH_ABBR[month - 1]} "
        f"{calendar_year % 100:02d}"
    )


def format_wasde_item(item):
    """Convert normalized item names into readable labels."""
    return str(item).replace("_", " ").strip().title()


# ============================================================
# 3. ANALYTICAL SECTIONS
# ============================================================

# Keys must match the normalized snd_item values in your CSV.
# Add or adjust aliases after checking your actual item names.

WASDE_SECTIONS = {
    "Area & Yield": [
        "area_planted",
        "area_harvested",
        "yield",
    ],

    "Supply": [
        "beginning_stocks",
        "production",
        "imports",
        "total_supply",
    ],

    "Demand & Usage": [
        "exports",
        "feed_and_residual",
        "feed_residual",
        "ethanol",
        "food_seed_and_industrial",
        "food_seed_industrial",
        "food_seed_and_other_uses",
        "crushings",
        "total_use",
    ],

    "Ending Stocks & Tightness": [
        "ending_stocks",
        "stocks_to_use",
        "ending_stocks_to_use",
        "stocks_use",
    ],
}


def split_wasde_sections(table):
    """
    Split a pivoted table into analytical sections.

    Input index: normalized snd_item values.
    Output: dictionary of section names and DataFrames.
    Unclassified rows are retained in Other.
    """
    sections = {}

    classified_items = {
        item
        for items in WASDE_SECTIONS.values()
        for item in items
    }

    for section, items in WASDE_SECTIONS.items():
        available = [
            item for item in items
            if item in table.index
        ]

        if available:
            sections[section] = table.loc[available].copy()

    other_items = [
        item for item in table.index
        if item not in classified_items
    ]

    if other_items:
        sections["Other"] = table.loc[other_items].copy()

    # Apply readable labels only after classification.
    for section_table in sections.values():
        section_table.index = [
            format_wasde_item(item)
            for item in section_table.index
        ]
        section_table.index.name = "WASDE Item"

    return sections


def display_sections(sections):
    """Display each section in a notebook."""
    for section, section_table in sections.items():
        print(f"\n{'=' * 15} {section} {'=' * 15}")
        display(section_table.round(2))


# ============================================================
# 4. SELECT DATA
# ============================================================

COMMODITY = "corn"
COUNTRY = "united states"
TABLE = "us"

data = df.loc[
    (df["commodity"] == COMMODITY)
    & (df["country"] == COUNTRY)
    & (df["table"] == TABLE)
].copy()

if data.empty:
    raise ValueError(
        "No data found for the selected commodity, country and table."
    )

print("\nSelected records:", len(data))
print("Marketing years:", data["crop_year"].nunique())
print("WASDE items:", data["snd_item"].nunique())

print("\nAvailable WASDE items:")
print(sorted(data["snd_item"].dropna().unique()))


# ============================================================
# 5. YEARLY TABLE
# ============================================================

# Choose an offset such as 0_10.
SELECTED_MONTH_OFFSET = "0_10"

available_offsets = sorted(
    data["month_offset"].dropna().unique(),
    key=lambda offset: tuple(
        map(int, str(offset).split("_"))
    ),
)

if SELECTED_MONTH_OFFSET not in available_offsets:
    raise ValueError(
        f"{SELECTED_MONTH_OFFSET} is unavailable. "
        f"Available offsets: {available_offsets}"
    )

yearly_data = data.loc[
    data["month_offset"] == SELECTED_MONTH_OFFSET
].copy()

# Keep the latest publication for each item and marketing year.
keys = ["snd_item", "crop_year", "month_offset"]

yearly_data = (
    yearly_data
    .sort_values("report_date")
    .drop_duplicates(subset=keys, keep="last")
)

# Do not silently choose between duplicate cells.
duplicates = yearly_data.duplicated(
    subset=["snd_item", "crop_year"],
    keep=False,
)

if duplicates.any():
    display(
        yearly_data.loc[
            duplicates,
            [
                "report_date",
                "snd_item",
                "crop_year",
                "crop",
                "value",
                "month_offset",
            ],
        ].sort_values(["snd_item", "crop_year"])
    )

    raise ValueError(
        "Duplicate yearly cells remain. Inspect the records above."
    )

yearly_table = yearly_data.pivot(
    index="snd_item",
    columns="crop_year",
    values="value",
)

yearly_table.columns = [
    format_marketing_year(year)
    for year in yearly_table.columns
]

yearly_table.columns.name = "Marketing Year"

# Sort years chronologically before sectioning.
yearly_table = yearly_table.reindex(
    columns=sorted(
        yearly_table.columns,
        key=lambda label: int(label.split("/")[0]),
    )
)

yearly_sections = split_wasde_sections(yearly_table)

print(
    f"\nYEARLY TABLE — {format_month_offset(SELECTED_MONTH_OFFSET, min(data['crop_year']))}"
)

display_sections(yearly_sections)


# ============================================================
# 6. MONTHLY TABLE
# ============================================================

SELECTED_CROP_YEAR = 2010

monthly_data = data.loc[
    data["crop_year"] == SELECTED_CROP_YEAR
].copy()

if monthly_data.empty:
    raise ValueError(
        "No data found for marketing year "
        f"{format_marketing_year(SELECTED_CROP_YEAR)}."
    )

# Keep the latest publication for each item and offset.
keys = ["snd_item", "crop_year", "month_offset"]

monthly_data = (
    monthly_data
    .sort_values("report_date")
    .drop_duplicates(subset=keys, keep="last")
)

# Detect duplicate cells after excluding crop distinctions.
duplicates = monthly_data.duplicated(
    subset=["snd_item", "month_offset"],
    keep=False,
)

if duplicates.any():
    display(
        monthly_data.loc[
            duplicates,
            [
                "report_date",
                "snd_item",
                "crop_year",
                "crop",
                "value",
                "month_offset",
            ],
        ].sort_values(["snd_item", "month_offset"])
    )

    raise ValueError(
        "Duplicate monthly cells remain. Inspect the records above."
    )

monthly_table = monthly_data.pivot(
    index="snd_item",
    columns="month_offset",
    values="value",
)

# Sort by offset year and month, not alphabetically.
ordered_offsets = sorted(
    monthly_table.columns,
    key=lambda offset: tuple(
        map(int, str(offset).split("_"))
    ),
)

monthly_table = monthly_table.reindex(
    columns=ordered_offsets
)

# Rename columns to MMM YY, e.g. May 10, Jan 11.
monthly_table.columns = [
    format_month_offset(offset, SELECTED_CROP_YEAR)
    for offset in ordered_offsets
]

monthly_table.columns.name = (
    "Calendar Month — Marketing Year "
    f"{format_marketing_year(SELECTED_CROP_YEAR)}"
)

monthly_sections = split_wasde_sections(monthly_table)

print(
    "\nMONTHLY TABLE — Marketing Year "
    f"{format_marketing_year(SELECTED_CROP_YEAR)}"
)

display_sections(monthly_sections)


Rows: 202960
Available commodities: ['barley', 'coarse grain', 'corn', 'cotton', 'feed grains', 'oats', 'soybean meal', 'soybean oil', 'soybeans', 'wheat']

Selected records: 5844
Marketing years: 18
WASDE items: 16

Available WASDE items:
['avg. farm price', 'beginning stocks', 'domestic total', 'ending stocks', 'ethanol', 'exports', 'feed residual', 'food seed industrial', 'harvested area', 'imports', 'planted area', 'production', 'stock to use', 'supply total', 'use total', 'yield']

YEARLY TABLE — Oct 09

=============== Area & Yield ===============


Marketing Year,2010/11,2011/12,2012/13,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25
WASDE Item,,,,,,,,,,,,,,
Yield,155.8,148.1,122.0,174.2,168.0,173.4,171.8,180.7,168.4,178.4,176.5,171.9,173.0,183.8



=============== Supply ===============


Marketing Year,2010/11,2011/12,2012/13,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25
WASDE Item,,,,,,,,,,,,,,
Production,12664.0,12433.0,10706.0,14475.0,13555.0,15057.0,14280.0,14778.0,13779.0,14722.0,15019.0,13895.0,15064.0,15203.0
Imports,10.0,15.0,75.0,25.0,30.0,50.0,50.0,50.0,50.0,25.0,25.0,50.0,25.0,25.0



=============== Demand & Usage ===============


Marketing Year,2010/11,2011/12,2012/13,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25
WASDE Item,,,,,,,,,,,,,,
Exports,2000.0,1600.0,1150.0,1750.0,1850.0,2225.0,1850.0,2475.0,1900.0,2325.0,2500.0,2150.0,2025.0,2325.0
Ethanol,NaN,5000.0,4500.0,5125.0,5250.0,5275.0,5475.0,5650.0,5400.0,5050.0,5200.0,5275.0,5300.0,5450.0


... (6 outputs left out)


=============== Demand & Usage ===============


Calendar Month — Marketing Year 2010/11,May 10,Jun 10,Jul 10,Aug 10,Sep 10,Oct 10,Nov 10,Dec 10,Jan 11,Feb 11,...,Jul 11,Aug 11,Sep 11,Oct 11,Nov 11,Dec 11,Jan 12,Feb 12,Mar 12,Apr 12
WASDE Item,,,,,,,,,,,,,,,,,,,,,
Exports,2000.0,2000.0,1950.0,2050.0,2100.0,2000.0,1950.0,1950.0,1950.0,1950.0,...,1875.0,1825.0,1835.0,1835.0,1835.0,1835.0,1835.0,1835.0,1835.0,1835.0
Ethanol,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5050.0,5020.0,5020.0,5020.0,5021.0,5021.0,5021.0,5021.0,5021.0,5021.0



=============== Other ===============


Calendar Month — Marketing Year 2010/11,May 10,Jun 10,Jul 10,Aug 10,Sep 10,Oct 10,Nov 10,Dec 10,Jan 11,Feb 11,...,Jul 11,Aug 11,Sep 11,Oct 11,Nov 11,Dec 11,Jan 12,Feb 12,Mar 12,Apr 12
WASDE Item,,,,,,,,,,,,,,,,,,,,,
Beginning Stocks,1738.00,1603.00,1478.0,1426.0,1386.00,1708.00,1708.00,1708.00,1708.00,1708.00,...,1708.00,1708.00,1708.00,1708.00,1708.00,1708.00,1708.00,1708.00,1708.00,1708.00
Domestic Total,11300.00,11410.00,11410.0,11440.0,11340.00,11480.00,11480.00,11480.00,11480.00,11550.00,...,11430.00,11420.00,11430.00,11218.00,11220.00,11220.00,11220.00,11220.00,11220.00,11220.00
Ending Stocks,1818.00,1573.00,1373.0,1312.0,1116.00,902.00,827.00,832.00,745.00,675.00,...,880.00,940.00,920.00,1128.00,1128.00,1128.00,1128.00,1128.00,1128.00,1128.00
Feed Residual,5350.00,5350.00,5350.0,5350.0,5250.00,5400.00,5300.00,5300.00,5200.00,5200.00,...,5000.00,5000.00,5000.00,4803.00,4792.00,4792.00,4793.00,4793.00,4793.00,4793.00
Food Seed Industrial,5950.00,6060.00,6060.0,6090.0,6090.00,6080.00,6180.00,6180.00,6280.00,6350.00,...,6430.00,6420.00,6430.00,6415.00,6428.00,6428.00,6428.00,6428.00,6428.00,6428.00
Harvested Area,81.80,81.80,81.0,81.0,81.00,81.30,81.30,81.30,81.40,81.40,...,81.40,81.40,81.40,81.40,81.40,81.40,81.40,81.40,81.40,81.40
Planted Area,88.80,88.80,87.9,87.9,87.90,88.20,88.20,88.20,88.20,88.20,...,88.20,88.20,88.20,88.20,88.20,88.20,88.20,88.20,88.20,88.20
Stock To Use,0.14,0.12,0.1,0.1,0.08,0.07,0.06,0.06,0.06,0.05,...,0.07,0.07,0.07,0.09,0.09,0.09,0.09,0.09,0.09,0.09
Supply Total,15118.00,14983.00,14733.0,14802.0,14556.00,14382.00,14257.00,14262.00,14175.00,14175.00,...,14185.00,14185.00,14185.00,14182.00,14182.00,14182.00,14182.00,14182.00,14182.00,14182.00


In [35]:

# Inspect the actual WASDE items for your selected dataset


# Inspect the actual WASDE items for your selected dataset

items = (
    data[["snd_item"]]
    .drop_duplicates()
    .sort_values("snd_item")
    .reset_index(drop=True)
)

print(f"Total unique WASDE items: {len(items)}")

for i, item in enumerate(items["snd_item"], start=1):
    print(f"{i:02d}. {item}")

items = (
    data[["snd_item"]]
    .drop_duplicates()
    .sort_values("snd_item")
    .reset_index(drop=True)
)

print(f"Total unique WASDE items: {len(items)}")

for i, item in enumerate(items["snd_item"], start=1):
    print(f"{i:02d}. {item}")


Total unique WASDE items: 16
01. avg. farm price
02. beginning stocks
03. domestic total
04. ending stocks
05. ethanol
06. exports
07. feed residual
08. food seed industrial
09. harvested area
10. imports
11. planted area
12. production
13. stock to use
14. supply total
15. use total
16. yield
Total unique WASDE items: 16
01. avg. farm price
02. beginning stocks
03. domestic total
04. ending stocks
05. ethanol
06. exports
07. feed residual
08. food seed industrial
09. harvested area
10. imports
11. planted area
12. production
13. stock to use
14. supply total
15. use total
16. yield


In [37]:

# Inspect the actual WASDE items for your selected dataset

items = (
    data[["snd_item"]]
    .drop_duplicates()
    .sort_values("snd_item")
    .reset_index(drop=True)
)

print(f"Total unique WASDE items: {len(items)}")

for i, item in enumerate(items["snd_item"], start=1):
    print(f"{i:02d}. {item}")


Total unique WASDE items: 16
01. avg. farm price
02. beginning stocks
03. domestic total
04. ending stocks
05. ethanol
06. exports
07. feed residual
08. food seed industrial
09. harvested area
10. imports
11. planted area
12. production
13. stock to use
14. supply total
15. use total
16. yield
